# Africa Tiny U-Net V6.1 — Kitsune Ground-Side Training Notebook

**Purpose:** train and validate the lightweight CNN model on Landsat-8 thermal data, then fine-tune it against manual ActiveFire annotations. The resulting model is intended as the **ground-trained CNN artifact** for later Kitsune onboard integration.

### This V6.1 deliberately does NOT claim to simulate the final Kitsune optical system.

Primary experiment:

`Landsat-8 B10 → preprocessing → Tiny U-Net → F1/IoU/Precision/Recall → manual fine-tuning → ONNX/INT8`

The later flight/deployment validation can use PyRawS/THRAWS as an independent raw/cross-sensor robustness experiment and should be reported separately from true Kitsune LWIR validation.

## 1. Environment and experiment configuration

Run this first. The only persistent files written to `/kaggle/working` are experiment artifacts; source data lives in `/kaggle/tmp/fireedge`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/aminekacem721/thermal-edge-fire/thermal-edge-fire"
)
SRC_DIR = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)
print("Source dir:", SRC_DIR)
print("fireedge package:", SRC_DIR / "fireedge")

from fireedge.data.dataset import ActiveFireDataset

print("✅ fireedge import OK")

Project root: /kaggle/input/datasets/aminekacem721/thermal-edge-fire/thermal-edge-fire
Source dir: /kaggle/input/datasets/aminekacem721/thermal-edge-fire/thermal-edge-fire/src
fireedge package: /kaggle/input/datasets/aminekacem721/thermal-edge-fire/thermal-edge-fire/src/fireedge
✅ fireedge import OK


In [2]:
# ================================================================
# CELL 2 — ENVIRONMENT / PATHS / CONFIGURATION
# ================================================================

!pip -q install gdown rasterio tqdm scipy pandas onnx onnxruntime onnxsim onnxscript

import os
import sys
import gc
import json
import math
import random
import re
import shutil
import struct
import tempfile
import time
import zipfile
import zlib
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import transform as rio_transform
from scipy import ndimage
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch.nn.functional as F

SEED = 42
BASELINE_F1 = 0.40

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

WORK_ROOT = Path('/kaggle/working')
SCRATCH_ROOT = Path('/kaggle/tmp/fireedge')
OUTPUT_DIR = WORK_ROOT / 'africa_tiny_unet_v6_1'
DATA_ROOT = SCRATCH_ROOT / 'africa'

WORK_ROOT.mkdir(parents=True, exist_ok=True)
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Attach your project repo/Kaggle Dataset containing src/fireedge/.
PROJECT_ROOT = Path(
    "/kaggle/input/datasets/aminekacem721/thermal-edge-fire/thermal-edge-fire"
)

SRC_DIR = PROJECT_ROOT / "src"

# Make fireedge importable
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if not (PROJECT_ROOT / 'src' / 'fireedge').exists():
    candidates = [Path('/kaggle/working/thermal-edge-fire'), Path.cwd()]
    for candidate in candidates:
        if (candidate / 'src' / 'fireedge').exists():
            PROJECT_ROOT = candidate
            break

if not (PROJECT_ROOT / 'src' / 'fireedge').exists():
    raise FileNotFoundError(
        'Could not find src/fireedge/. Attach the thermal-edge-fire project as a Kaggle Dataset '
        'or change PROJECT_ROOT in this cell.'
    )

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Primary deadline-mode settings.
BATCH_SIZE = 8 if device.type == 'cuda' else 2
NUM_WORKERS = 2 if device.type == 'cuda' else 0
PIN_MEMORY = device.type == 'cuda'
EPOCHS = 20
PATIENCE = 5
POS_WEIGHT_CANDIDATES = [10.0, 15.0, 25.0]

# Fixed physical normalization used by the existing project contract.
NORM_MIN_K = 270.0
NORM_MAX_K = 420.0

# IMPORTANT: this is a Landsat training experiment, not the final Kitsune sensor simulation.
USE_MICROBOLOMETER_AUGMENTATION = False

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("fireedge:", SRC_DIR / "fireedge")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"PROJECT_ROOT does not exist:\n{PROJECT_ROOT}"
    )

if not (SRC_DIR / "fireedge").exists():
    raise FileNotFoundError(
        f"Could not find src/fireedge/ under:\n{PROJECT_ROOT}"
    )

print("✅ Project root found")
print("✅ src/fireedge found")
print("✅ Python source path configured")
print('OUTPUT_DIR:', OUTPUT_DIR)
print('SCRATCH_ROOT:', SCRATCH_ROOT)
print('DEVICE:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('BASELINE_F1:', BASELINE_F1)
print('Storage:')
!df -h /kaggle/working
!df -h /kaggle/tmp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 75.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 8.8 MB/s eta 0:00:00
PROJECT_ROOT: /kaggle/input/datasets/aminekacem721/thermal-edge-fire/thermal-edge-fire
SRC_DIR: /kaggle/input/datasets/aminekacem721/thermal-edge-fire/thermal-edge-fire/src
fireedge: /kaggle/input/datasets/aminekacem721/thermal-edge-fire/thermal-edge-fire/src/fireedge
✅ Project root found
✅ src/fireedge found
✅ Python source path configured
OUTPUT_DIR: /kaggle/working/africa_tiny_unet_v6_1
SCRATCH_ROOT: /kaggle/tmp/fireedge
DEVICE: cuda
GPU: Tesla T4
BASELINE_F1: 0.4
Storage:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G  144K   20G   1% /kaggle/working
Filesystem      Size  Used Avail Use% Mounted on
overlay         8.0T  6.9T  1.1T 

## 2. Download only the African archive

The archive used here is the African regional `patches_and_masks/Africa.zip`, not the complete Google Drive root.

In [3]:
# ================================================================
# CELL 3 — DOWNLOAD AFRICA.ZIP ONLY
# ================================================================

import gdown
from pathlib import Path

AFRICA_FILE_ID = "1Ng3JwsjJPApshk8lJdGcsNHI52NaEMDX"
AFRICA_ARCHIVE = DATA_ROOT / "Africa.zip"

DATA_ROOT.mkdir(parents=True, exist_ok=True)

if AFRICA_ARCHIVE.exists():
    size_gb = AFRICA_ARCHIVE.stat().st_size / (1024 ** 3)
    if size_gb < 0.01:
        print("Existing Africa.zip looks invalid/suspicious; removing it.")
        AFRICA_ARCHIVE.unlink()
    else:
        print(f"Africa.zip already exists: {size_gb:.2f} GB")

if not AFRICA_ARCHIVE.exists():
    print("Downloading Africa.zip to /kaggle/tmp ...")

    gdown.download(
        id=AFRICA_FILE_ID,
        output=str(AFRICA_ARCHIVE),
        quiet=False,
    )

assert AFRICA_ARCHIVE.exists(), "Africa.zip was not downloaded."

archive_size_gb = AFRICA_ARCHIVE.stat().st_size / (1024 ** 3)

assert archive_size_gb > 0.01, (
    f"Africa.zip appears too small: {archive_size_gb:.4f} GB"
)

print()
print(f"✅ Africa.zip downloaded")
print(f"Path: {AFRICA_ARCHIVE}")
print(f"Size: {archive_size_gb:.2f} GB")

print("\nScratch storage:")
!du -sh /kaggle/tmp/fireedge 2>/dev/null || true

Downloading...
From (original): https://drive.google.com/uc?id=1Ng3JwsjJPApshk8lJdGcsNHI52NaEMDX
From (redirected): https://drive.google.com/uc?id=1Ng3JwsjJPApshk8lJdGcsNHI52NaEMDX&confirm=t&uuid=81198b3e-3148-45fd-811d-56f24ea4ba9b
To: /kaggle/tmp/fireedge/africa/Africa.zip
100%|██████████| 31.7G/31.7G [04:51<00:00, 109MB/s] 



✅ Africa.zip downloaded
Path: /kaggle/tmp/fireedge/africa/Africa.zip
Size: 29.49 GB

Scratch storage:
30G	/kaggle/tmp/fireedge


In [4]:
# ================================================================
# CELL 4 — INSPECT THE AFRICA ARCHIVE STRUCTURE
# ================================================================

import re
import zipfile
from pathlib import Path

with zipfile.ZipFile(AFRICA_ARCHIVE, "r") as archive:
    members = archive.namelist()

print(f"Total members in Africa.zip: {len(members):,}")

# ----------------------------------------------------------------
# Identify ONLY real scene image archives:
#   z166052.zip
#   z166053.zip
# etc.
#
# Explicitly excludes:
#   z166052_masks.zip
#   z166052_masks_derivates.zip
#   anything else
# ----------------------------------------------------------------

image_scene_archives = []

for member in members:
    name = Path(member).name

    if re.fullmatch(r"z\d+\.zip", name, flags=re.IGNORECASE):
        image_scene_archives.append(member)

# ----------------------------------------------------------------
# Identify ONLY matching mask archives:
#   z166052_masks.zip
# ----------------------------------------------------------------

mask_scene_archives = []

for member in members:
    name = Path(member).name

    if re.fullmatch(r"z\d+_masks\.zip", name, flags=re.IGNORECASE):
        mask_scene_archives.append(member)

image_scene_archives = sorted(image_scene_archives)
mask_scene_archives = sorted(mask_scene_archives)

# ----------------------------------------------------------------
# Scene keys
# ----------------------------------------------------------------

def image_scene_key(member):
    """
    z166052.zip -> z166052
    """
    return Path(member).stem.lower()


def mask_scene_key(member):
    """
    z166052_masks.zip -> z166052
    """
    name = Path(member).name.lower()
    return re.sub(r"_masks\.zip$", "", name, flags=re.IGNORECASE)


image_scene_keys = {
    image_scene_key(m)
    for m in image_scene_archives
}

mask_scene_keys = {
    mask_scene_key(m)
    for m in mask_scene_archives
}

matched_scene_keys = sorted(image_scene_keys & mask_scene_keys)
image_only_scene_keys = sorted(image_scene_keys - mask_scene_keys)
mask_only_scene_keys = sorted(mask_scene_keys - image_scene_keys)

# ----------------------------------------------------------------
# Report
# ----------------------------------------------------------------

print()
print("========== AFRICA ARCHIVE STRUCTURE ==========")
print(f"Real image scene archives:       {len(image_scene_archives):,}")
print(f"Real mask scene archives:        {len(mask_scene_archives):,}")
print(f"Matched image+mask scenes:       {len(matched_scene_keys):,}")
print(f"Images without matching mask:    {len(image_only_scene_keys):,}")
print(f"Masks without matching image:    {len(mask_only_scene_keys):,}")

print()
print("Examples of IMAGE archives:")
for m in image_scene_archives[:10]:
    print("  ", m)

print()
print("Examples of MASK archives:")
for m in mask_scene_archives[:10]:
    print("  ", m)

# ----------------------------------------------------------------
# Confirm that derivative archives are NOT being treated as images
# ----------------------------------------------------------------

derivative_archives = [
    m for m in members
    if Path(m).name.lower().endswith("_masks_derivates.zip")
]

print()
print(f"Excluded *_masks_derivates.zip archives: {len(derivative_archives):,}")

assert image_scene_archives, "No real image scene archives found."
assert mask_scene_archives, "No real mask scene archives found."
assert matched_scene_keys, "No image/mask scene pairs found."

print()
print("✅ Archive structure looks valid.")
print("✅ Derivative mask archives are excluded.")
print("✅ Matched scenes identified.")

Total members in Africa.zip: 1,475

========== AFRICA ARCHIVE STRUCTURE ==========
Real image scene archives:       513
Real mask scene archives:        513
Matched image+mask scenes:       513
Images without matching mask:    0
Masks without matching image:    0

Examples of IMAGE archives:
   z152074.zip
   z153074.zip
   z153075.zip
   z154074.zip
   z154075.zip
   z156063.zip
   z158069.zip
   z158070.zip
   z158071.zip
   z158072.zip

Examples of MASK archives:
   z152074_masks.zip
   z153074_masks.zip
   z153075_masks.zip
   z154074_masks.zip
   z154075_masks.zip
   z156063_masks.zip
   z158069_masks.zip
   z158070_masks.zip
   z158071_masks.zip
   z158072_masks.zip

Excluded *_masks_derivates.zip archives: 449

✅ Archive structure looks valid.
✅ Derivative mask archives are excluded.
✅ Matched scenes identified.


## 3. Extract every African patch and its masks

No `MAX_PATCHES`, no `MAX_PATCHES_PER_SCENE`. Every image patch in the Africa regional archive is indexed.

In [5]:
# ================================================================
# CELL 5 — EXTRACT ONLY MATCHED AFRICAN PATCHES + ALGORITHM MASKS
# ================================================================

import shutil
import tempfile
import zipfile
import re
from pathlib import Path

from tqdm.auto import tqdm

# ----------------------------------------------------------------
# Output directories
# ----------------------------------------------------------------

PREPARED_ROOT = DATA_ROOT / "africa_prepared"
PATCH_DIR = PREPARED_ROOT / "patches"
ALGORITHM_MASK_DIR = PREPARED_ROOT / "masks" / "algorithmic"

# Clean any previous partial extraction from this session.
if PREPARED_ROOT.exists():
    print(f"Removing previous prepared data: {PREPARED_ROOT}")
    shutil.rmtree(PREPARED_ROOT)

PATCH_DIR.mkdir(parents=True, exist_ok=True)
ALGORITHM_MASK_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------
# Build lookup tables from CELL 4
# ----------------------------------------------------------------

image_member_by_scene = {
    image_scene_key(member): member
    for member in image_scene_archives
}

mask_member_by_scene = {
    mask_scene_key(member): member
    for member in mask_scene_archives
}

matched_scene_keys = sorted(
    set(image_member_by_scene) & set(mask_member_by_scene)
)

print(f"Scenes to extract: {len(matched_scene_keys):,}")

# ----------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------

def canonical_patch_name(name):
    """
    Converts algorithm-specific mask names back to the base patch stem.

    Example:
        abc_p01_kumar-roy.tif -> abc_p01
        abc_p01_murphy.tif    -> abc_p01
        abc_p01_schroeder.tif -> abc_p01
        abc_p01_voting.tif    -> abc_p01
    """
    stem = Path(name).stem

    stem = re.sub(
        r"_(?:kumar[-_ ]?roy|kumarroy|murphy|schroeder)"
        r"(?=_p|$)",
        "",
        stem,
        flags=re.IGNORECASE,
    )

    stem = re.sub(
        r"_(?:voting|intersection|mask)"
        r"(?=_p|$)",
        "",
        stem,
        flags=re.IGNORECASE,
    )

    return stem


def detect_algorithm(name):
    low = Path(name).name.lower()

    if "kumar" in low and "roy" in low:
        return "kumar_roy"

    if "murphy" in low:
        return "murphy"

    if "schroeder" in low:
        return "schroeder"

    return None


def open_nested_zip(outer_archive, member):
    """
    Open one nested ZIP without extracting the entire outer archive.

    The temporary file is deleted automatically when closed.
    """

    temp_file = tempfile.TemporaryFile()

    with zipfile.ZipFile(outer_archive, "r") as outer:
        with outer.open(member) as src:
            shutil.copyfileobj(src, temp_file, length=16 * 1024 * 1024)

    temp_file.seek(0)

    nested = zipfile.ZipFile(temp_file, "r")

    return temp_file, nested


# ----------------------------------------------------------------
# 1. Extract ONLY image TIFFs from matched scene archives
# ----------------------------------------------------------------

image_names = []
image_stems = set()
duplicate_images = []

with zipfile.ZipFile(AFRICA_ARCHIVE, "r"):

    for scene in tqdm(
        matched_scene_keys,
        desc="Extracting matched image scenes"
    ):

        member = image_member_by_scene[scene]

        temp_file, nested = open_nested_zip(
            AFRICA_ARCHIVE,
            member
        )

        try:
            for info in nested.infolist():

                if info.is_dir():
                    continue

                if not info.filename.lower().endswith((".tif", ".tiff")):
                    continue

                name = Path(info.filename).name
                stem = Path(name).stem

                destination = PATCH_DIR / name

                # Protect against silent overwriting.
                if destination.exists():
                    duplicate_images.append(name)
                    continue

                with nested.open(info) as src, open(destination, "wb") as dst:
                    shutil.copyfileobj(
                        src,
                        dst,
                        length=16 * 1024 * 1024
                    )

                image_names.append(name)
                image_stems.add(stem)

        finally:
            nested.close()
            temp_file.close()


# ----------------------------------------------------------------
# 2. Extract ONLY the 3 algorithmic masks used for voting
# ----------------------------------------------------------------

saved_masks = 0
mask_names = []

with zipfile.ZipFile(AFRICA_ARCHIVE, "r"):

    for scene in tqdm(
        matched_scene_keys,
        desc="Extracting algorithmic masks"
    ):

        member = mask_member_by_scene[scene]

        temp_file, nested = open_nested_zip(
            AFRICA_ARCHIVE,
            member
        )

        try:
            for info in nested.infolist():

                if info.is_dir():
                    continue

                if not info.filename.lower().endswith((".tif", ".tiff")):
                    continue

                name = Path(info.filename).name

                algorithm = detect_algorithm(name)

                # Ignore intersection/voting/derivative products here.
                if algorithm is None:
                    continue

                target = canonical_patch_name(name)

                # Keep only masks corresponding to extracted images.
                if target not in image_stems:
                    continue

                destination = ALGORITHM_MASK_DIR / name

                if destination.exists():
                    continue

                with nested.open(info) as src, open(destination, "wb") as dst:
                    shutil.copyfileobj(
                        src,
                        dst,
                        length=16 * 1024 * 1024
                    )

                saved_masks += 1
                mask_names.append(name)

        finally:
            nested.close()
            temp_file.close()


# ----------------------------------------------------------------
# 3. Summary
# ----------------------------------------------------------------

patch_count = len(list(PATCH_DIR.glob("*.tif")))
patch_count += len(list(PATCH_DIR.glob("*.tiff")))

mask_count = len(list(ALGORITHM_MASK_DIR.glob("*.tif")))
mask_count += len(list(ALGORITHM_MASK_DIR.glob("*.tiff")))

print()
print("========== EXTRACTION COMPLETE ==========")
print(f"Matched scenes:             {len(matched_scene_keys):,}")
print(f"Image patches extracted:    {patch_count:,}")
print(f"Algorithm masks extracted: {mask_count:,}")

if duplicate_images:
    print()
    print(
        f"⚠️ Duplicate image filenames skipped: "
        f"{len(duplicate_images):,}"
    )
else:
    print("✅ No duplicate image filenames detected.")

print()
print("Prepared data:")
print(f"  patches: {PATCH_DIR}")
print(f"  masks:   {ALGORITHM_MASK_DIR}")

print()
print("Scratch usage:")
!du -sh /kaggle/tmp/fireedge 2>/dev/null || true

Scenes to extract: 513


Extracting matched image scenes:   0%|          | 0/513 [00:00<?, ?it/s]

Extracting algorithmic masks:   0%|          | 0/513 [00:00<?, ?it/s]


========== EXTRACTION COMPLETE ==========
Matched scenes:             513
Image patches extracted:    27,729
Algorithm masks extracted: 68,005
✅ No duplicate image filenames detected.

Prepared data:
  patches: /kaggle/tmp/fireedge/africa/africa_prepared/patches
  masks:   /kaggle/tmp/fireedge/africa/africa_prepared/masks/algorithmic

Scratch usage:
68G	/kaggle/tmp/fireedge


In [6]:
from pathlib import Path
import shutil

def size_gb(path):
    total = 0
    path = Path(path)
    if path.exists():
        for p in path.rglob("*"):
            if p.is_file():
                try:
                    total += p.stat().st_size
                except OSError:
                    pass
    return total / (1024**3)

print("=== SIZES ===")
print(f"/kaggle/tmp/fireedge:      {size_gb('/kaggle/tmp/fireedge'):.2f} GB")
print(f"/kaggle/working:            {size_gb('/kaggle/working'):.2f} GB")

print("\n=== TOP LEVEL /kaggle/tmp/fireedge ===")
for p in Path("/kaggle/tmp/fireedge").iterdir():
    if p.is_dir():
        print(f"{p.name:30s} {size_gb(p):8.2f} GB")
    else:
        print(f"{p.name:30s} {p.stat().st_size/(1024**3):8.2f} GB")

=== SIZES ===
/kaggle/tmp/fireedge:      67.58 GB
/kaggle/working:            0.00 GB

=== TOP LEVEL /kaggle/tmp/fireedge ===
africa                            67.58 GB


In [7]:
# ================================================================
# CELL 6 — BUILD THE OFFICIAL VOTING TARGET FOR EVERY IMAGE
# ================================================================

# The published ActiveFire voting target requires >=2 methods to agree.
# Therefore an image with zero/one detected algorithmic mask gets a zero
# voting target; it is a voting-negative, not a missing training sample.

VOTING_DIR = PREPARED_ROOT / 'masks' / 'voting'
VOTING_DIR.mkdir(parents=True, exist_ok=True)

image_files = sorted(list(PATCH_DIR.glob('*.tif')) + list(PATCH_DIR.glob('*.tiff')))

masks_by_image = defaultdict(dict)
for mask_file in ALGORITHM_MASK_DIR.glob('*.tif'):
    alg = detect_algorithm(mask_file.name)
    target = canonical_patch_name(mask_file.name)
    if alg is not None:
        masks_by_image[target][alg] = mask_file

records = []
coverage = []

for image_file in tqdm(image_files, desc='Building voting masks'):
    stem = image_file.stem
    sources = masks_by_image.get(stem, {})
    coverage.append({
        'stem': stem,
        'n_algorithm_masks': len(sources),
    })

    with rasterio.open(image_file) as img_src:
        profile = img_src.profile.copy()
        height, width = img_src.height, img_src.width

    arrays = []
    for alg in ('kumar_roy', 'murphy', 'schroeder'):
        path = sources.get(alg)
        if path is None:
            continue
        with rasterio.open(path) as mask_src:
            arr = (mask_src.read(1) > 0).astype(np.uint8)
            if arr.shape != (height, width):
                raise ValueError(
                    f'Shape mismatch for {stem}: {arr.shape} vs {(height, width)}'
                )
            arrays.append(arr)

    if arrays:
        vote = (
            np.sum(arrays, axis=0) >= 2
        ).astype(np.uint8)
    else:
        vote = np.zeros((height, width), dtype=np.uint8)

    for k in ('blockxsize', 'blockysize', 'tiled'):
        profile.pop(k, None)
    profile.update(
        driver='GTiff',
        count=1,
        dtype='uint8',
        nodata=0,
        compress='lzw',
    )

    output = VOTING_DIR / f'{stem}_voting.tif'
    with rasterio.open(output, 'w', **profile) as dst:
        dst.write(vote, 1)

    records.append({
        'stem': stem,
        'image_path': str(image_file),
        'mask_path': str(output),
        'fire_pixels': int(vote.sum()),
        'n_algorithm_masks': len(arrays),
        'sample_type': 'voting_fire' if vote.any() else 'voting_zero',
    })

coverage_df = pd.DataFrame(coverage)
africa_df = pd.DataFrame(records)

assert len(africa_df) == len(image_files)

africa_df.to_csv(
    PREPARED_ROOT / 'africa_voting_inventory.csv',
    index=False,
)

print('Total African patches:', len(africa_df))
print('\nLabel distribution:')
display(africa_df['sample_type'].value_counts())
print('\nAlgorithm-mask coverage:')
display(coverage_df['n_algorithm_masks'].value_counts().sort_index())
print('\nFire-pixel statistics:')
display(africa_df['fire_pixels'].describe())

Building voting masks:   0%|          | 0/27729 [00:00<?, ?it/s]

Total African patches: 27729

Label distribution:


sample_type
voting_fire    22500
voting_zero     5229
Name: count, dtype: int64


Algorithm-mask coverage:


n_algorithm_masks
1     5138
2     4906
3    17685
Name: count, dtype: int64


Fire-pixel statistics:


count    27729.000000
mean         8.854376
std         24.357180
min          0.000000
25%          1.000000
50%          3.000000
75%          8.000000
max        630.000000
Name: fire_pixels, dtype: float64

## 4. Scene-level train/validation/test split

The split is performed at acquisition-scene level so adjacent patches from one Landsat scene cannot leak between train and test.

In [8]:
# ================================================================
# CELL 7 — SCENE-LEVEL SPLIT
# ================================================================

df_all = africa_df.copy()

df_all['parent_scene'] = (
    df_all['stem']
    .str.replace(r'_p\d+$', '', regex=True)
)

rng = np.random.default_rng(SEED)
scenes = df_all['parent_scene'].drop_duplicates().to_numpy()
assert len(scenes) >= 3, (
    f'Need at least 3 independent scenes for train/val/test; found {len(scenes)}.'
)
rng.shuffle(scenes)

n_test = max(1, round(0.15 * len(scenes)))
n_val = max(1, round(0.15 * len(scenes)))

# Keep at least one independent scene in the training split even for small releases.
while n_test + n_val >= len(scenes):
    if n_val > 1:
        n_val -= 1
    elif n_test > 1:
        n_test -= 1
    else:
        raise RuntimeError('Unable to create non-empty train/val/test scene splits.')

test_scenes = set(scenes[:n_test])
val_scenes = set(scenes[n_test:n_test + n_val])
train_scenes = set(scenes[n_test + n_val:])

assert train_scenes and val_scenes and test_scenes
assert not (train_scenes & val_scenes)
assert not (train_scenes & test_scenes)
assert not (val_scenes & test_scenes)


def split_of(scene):
    if scene in train_scenes:
        return 'train'
    if scene in val_scenes:
        return 'val'
    return 'test'


df_all['split'] = df_all['parent_scene'].map(split_of)

train_df = df_all[df_all['split'] == 'train'].copy()
val_df = df_all[df_all['split'] == 'val'].copy()
test_df = df_all[df_all['split'] == 'test'].copy()

print(f'Train: {len(train_df):,} patches / {train_df.parent_scene.nunique():,} scenes')
print(f'Val:   {len(val_df):,} patches / {val_df.parent_scene.nunique():,} scenes')
print(f'Test:  {len(test_df):,} patches / {test_df.parent_scene.nunique():,} scenes')
print(f'Approximate scene split: {len(train_scenes)/len(scenes):.1%} / {len(val_scenes)/len(scenes):.1%} / {len(test_scenes)/len(scenes):.1%}')

assert len(train_df) and len(val_df) and len(test_df)

split_summary = (
    df_all.groupby(['split', 'sample_type'])
    .size()
    .unstack(fill_value=0)
)
display(split_summary)

split_summary.to_csv(OUTPUT_DIR / 'split_summary_v6_1.csv')
df_all.to_csv(OUTPUT_DIR / 'africa_inventory_split_v6_1.csv', index=False)


Train: 18,931 patches / 360 scenes
Val:   4,017 patches / 77 scenes
Test:  4,781 patches / 77 scenes
Approximate scene split: 70.0% / 15.0% / 15.0%


sample_type,voting_fire,voting_zero
split,,
test,3946,835
train,15377,3554
val,3177,840


In [12]:
# ================================================================
# CELL 8 — INPUT / MASK SANITY CHECK
# ================================================================

from fireedge.data.dataset import ActiveFireDataset

# One real sample from each split.
for name, frame in [('train', train_df), ('val', val_df), ('test', test_df)]:
    row = frame.iloc[0]
    with rasterio.open(row.image_path) as src:
        raw = src.read()
    with rasterio.open(row.mask_path) as src:
        mask = src.read(1)

    print(f'[{name}]')
    print('  image shape:', raw.shape, 'dtype:', raw.dtype)
    print('  B10 raw min/max:', int(raw[8].min()), int(raw[8].max()))
    print('  B11 raw min/max:', int(raw[9].min()), int(raw[9].max()))
    print('  mask shape:', mask.shape, 'values:', np.unique(mask))
    print()

# Dataset contract check.
sanity_ds = ActiveFireDataset(
    train_df.head(min(4, len(train_df))),
    channels=(8,),
    norm_method='fixed',
    calibrate_temperature=True,
    degrader=None,
    augment=False,
)

sample_x, sample_y = sanity_ds[0]
print('Dataset tensor:', tuple(sample_x.shape), sample_x.dtype)
print('Dataset mask:', tuple(sample_y.shape), sample_y.dtype)
print('Normalized range:', float(sample_x.min()), float(sample_x.max()))
print('Mask values:', torch.unique(sample_y))

assert sample_x.ndim == 3 and sample_x.shape[0] == 1
assert sample_y.ndim == 3 and sample_y.shape[0] == 1
assert torch.isfinite(sample_x).all()
assert torch.isfinite(sample_y).all()

[train]
  image shape: (10, 256, 256) dtype: uint16
  B10 raw min/max: 22256 30397
  B11 raw min/max: 21131 26676
  mask shape: (256, 256) values: [0 1]

[val]
  image shape: (10, 256, 256) dtype: uint16
  B10 raw min/max: 22399 29845
  B11 raw min/max: 21082 26576
  mask shape: (256, 256) values: [0 1]

[test]
  image shape: (10, 256, 256) dtype: uint16
  B10 raw min/max: 22251 28428
  B11 raw min/max: 21023 24929
  mask shape: (256, 256) values: [0 1]

Dataset tensor: (1, 256, 256) torch.float32
Dataset mask: (1, 256, 256) torch.float32
Normalized range: 0.09710205346345901 0.23036275804042816
Mask values: tensor([0., 1.])


## 5. Tiny U-Net and metrics

The primary experiment is now deliberately **native Landsat training**. The artificial `flight_domain_pair()` transformation is not used because its GSD transformation was only a proxy, not a validated Kitsune sensor model.

In [16]:
# ================================================================
# CELL 9 — DATALOADERS + TINY U-NET
# ================================================================

# Ground-side primary experiment: no unvalidated sensor-domain deformation.
train_ds = ActiveFireDataset(
    train_df,
    channels=(8,),
    norm_method='fixed',
    calibrate_temperature=True,
    degrader=None,
    augment=True,
    random_seed=SEED,
)

val_ds = ActiveFireDataset(
    val_df,
    channels=(8,),
    norm_method='fixed',
    calibrate_temperature=True,
    degrader=None,
    augment=False,
)

test_ds = ActiveFireDataset(
    test_df,
    channels=(8,),
    norm_method='fixed',
    calibrate_temperature=True,
    degrader=None,
    augment=False,
)

is_negative = (
    train_df['sample_type'] == 'voting_zero'
).to_numpy()
n_negative = int(is_negative.sum())
n_positive = int((~is_negative).sum())
assert n_positive > 0 and n_negative > 0

TARGET_NEGATIVE_SHARE = 0.35
negative_weight = (
    TARGET_NEGATIVE_SHARE * n_positive
    / ((1 - TARGET_NEGATIVE_SHARE) * n_negative)
)

sample_weights = np.where(is_negative, negative_weight, 1.0)
train_sampler = WeightedRandomSampler(
    torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_ds),
    replacement=True,
)

common_loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
if NUM_WORKERS > 0:
    common_loader_kwargs['persistent_workers'] = True

train_loader = DataLoader(
    train_ds,
    sampler=train_sampler,
    **common_loader_kwargs,
)
val_loader = DataLoader(
    val_ds,
    shuffle=False,
    **common_loader_kwargs,
)
test_loader = DataLoader(
    test_ds,
    shuffle=False,
    **common_loader_kwargs,
)


class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class TinyUNet(nn.Module):
    def __init__(self, base=8):
        super().__init__()
        self.e1 = ConvBlock(1, base)
        self.e2 = ConvBlock(base, base * 2)
        self.e3 = ConvBlock(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)
        self.mid = ConvBlock(base * 4, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.d3 = ConvBlock(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.d2 = ConvBlock(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.d1 = ConvBlock(base * 2, base)
        self.out = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        middle = self.mid(self.pool(e3))
        d3 = self.d3(torch.cat([self.u3(middle), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        return self.out(d1)


model_probe = TinyUNet(base=8)
parameter_count = sum(p.numel() for p in model_probe.parameters())
print('Tiny U-Net parameters:', parameter_count)
assert parameter_count < 200_000

xb, yb = next(iter(train_loader))
print('Input:', tuple(xb.shape), xb.dtype)
print('Mask:', tuple(yb.shape), yb.dtype)
print('Input range:', float(xb.min()), float(xb.max()))
print('Mask values:', torch.unique(yb))

Tiny U-Net parameters: 121033
Input: (8, 1, 256, 256) torch.float32
Mask: (8, 1, 256, 256) torch.float32
Input range: 0.0 0.2686081826686859
Mask values: tensor([0., 1.])


In [40]:
# ================================================================
# CELL 10 — LOSS / F1 / IoU / THRESHOLD SEARCH
# ================================================================

class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=0.75, eps=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.eps = eps

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        dims = (1, 2, 3)
        tp = (p * targets).sum(dims)
        fp = (p * (1 - targets)).sum(dims)
        fn = ((1 - p) * targets).sum(dims)
        tv = (tp + self.eps) / (
            tp + self.alpha * fp + self.beta * fn + self.eps
        )
        return ((1 - tv) ** self.gamma).mean()


def metric_counts(pred, target):
    pred = pred.bool()
    target = target.bool()
    return (
        int((pred & target).sum().item()),
        int((pred & ~target).sum().item()),
        int((~pred & target).sum().item()),
    )


def metrics_from_counts(tp, fp, fn):
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * tp / (2 * tp + fp + fn + 1e-9)
    iou = tp / (tp + fp + fn + 1e-9)
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'iou': iou,
        'tp': tp,
        'fp': fp,
        'fn': fn,
    }


@torch.no_grad()
def evaluate(model, loader, loss_fn, threshold, collect_frame_metrics=True):
    model.eval()
    total_loss = []
    tp = fp = fn = 0
    fire_frames = 0
    detected_fire_frames = 0
    correct_fire_frames = 0
    no_fire_frames = 0
    false_alarm_frames = 0

    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        logits = model(images)
        loss = loss_fn(logits, masks)
        total_loss.append(float(loss.item()))
        pred = torch.sigmoid(logits) >= threshold

        a, b, c = metric_counts(pred, masks)
        tp += a
        fp += b
        fn += c

        if collect_frame_metrics:
            for j in range(pred.shape[0]):
                gt_fire = bool(masks[j].any().item())
                pd_fire = bool(pred[j].any().item())
                fire_frames += int(gt_fire)
                no_fire_frames += int(not gt_fire)
                detected_fire_frames += int(pd_fire)
                correct_fire_frames += int(gt_fire and pd_fire)
                false_alarm_frames += int((not gt_fire) and pd_fire)

    out = metrics_from_counts(tp, fp, fn)
    out['loss'] = float(np.mean(total_loss))
    out['frame_recall'] = correct_fire_frames / max(fire_frames, 1)
    out['frame_false_alarm_rate'] = false_alarm_frames / max(no_fire_frames, 1)
    out['fire_frames'] = fire_frames
    out['no_fire_frames'] = no_fire_frames
    out['detected_fire_frames'] = detected_fire_frames
    return out


@torch.no_grad()
def threshold_sweep(model, loader, loss_fn, coarse=True):
    if coarse:
        thresholds = np.arange(0.05, 0.951, 0.05)   # was 0.501
    else:
        thresholds = np.arange(0.05, 0.951, 0.01)   # was 0.501
    rows = []
    for t in thresholds:
        m = evaluate(model, loader, loss_fn, float(t))
        rows.append({'threshold': float(t), **m})
    return pd.DataFrame(rows)


def choose_threshold(model, loader, loss_fn):
    coarse = threshold_sweep(model, loader, loss_fn, coarse=True)
    best_coarse = coarse.sort_values(['f1', 'precision'], ascending=False).iloc[0]
    center = float(best_coarse['threshold'])
    fine_thresholds = np.arange(
        max(0.05, center - 0.05),
        min(0.95, center + 0.0501) + 0.001,   # +epsilon makes the bound effectively inclusive
        0.01,
    )
    rows = []
    for t in fine_thresholds:
        m = evaluate(model, loader, loss_fn, float(round(t, 2)))
        rows.append({'threshold': float(round(t, 2)), **m})
    fine = pd.DataFrame(rows)
    best = fine.sort_values(['f1', 'precision'], ascending=False).iloc[0]
    return coarse, fine, best


def print_comparison(label, metrics):
    improvement = metrics['f1'] - BASELINE_F1
    relative = 100 * improvement / max(BASELINE_F1, 1e-9)
    print(f'{label}')
    print(f"  F1:        {metrics['f1']:.4f}")
    print(f"  IoU:       {metrics['iou']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  Frame recall: {metrics['frame_recall']:.4f}")
    print(f"  False-alarm frame rate: {metrics['frame_false_alarm_rate']:.4f}")
    print(f"  vs baseline 0.40: {improvement:+.4f} ({relative:+.1f}%)")

loss_tversky = FocalTverskyLoss()
THRESHOLDS = np.arange(0.05, 0.951, 0.05)

In [50]:
# ================================================================
# FINAL BASE MODEL — AFRICA TEST AT VALIDATION-SELECTED THRESHOLD
# ================================================================

test_eval_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

checkpoint_path = OUTPUT_DIR / "tiny_unet_b8_pw25_best.pt"

model = TinyUNet(base=8).to(device)

ckpt = torch.load(
    checkpoint_path,
    map_location=device,
)

model.load_state_dict(ckpt["model_state"])

BASE_TEST_THRESHOLD = 0.77

base_test_metrics = evaluate(
    model,
    test_eval_loader,
    loss_tversky,
    BASE_TEST_THRESHOLD,
)

print("BASE MODEL — FINAL AFRICA TEST")
print(f"Threshold:   {BASE_TEST_THRESHOLD:.2f}")
print(f"F1:          {base_test_metrics['f1']:.4f}")
print(f"IoU:         {base_test_metrics['iou']:.4f}")
print(f"Precision:   {base_test_metrics['precision']:.4f}")
print(f"Recall:      {base_test_metrics['recall']:.4f}")
print(f"Frame recall:{base_test_metrics['frame_recall']:.4f}")
print(
    f"False-alarm frame rate: "
    f"{base_test_metrics['frame_false_alarm_rate']:.4f}"
)

BASE MODEL — FINAL AFRICA TEST
Threshold:   0.77
F1:          0.4076
IoU:         0.2560
Precision:   0.5175
Recall:      0.3362
Frame recall:0.3011
False-alarm frame rate: 0.1174


## 6. Base training

Deadline-mode uses two positive-class weights (`10` and `25`) and 20 epochs with early stopping. This is enough to determine whether the corrected full-Africa methodology materially improves the ~0.40 F1 baseline.

In [21]:
# ================================================================
# CELL 11 — BASE TRAINING / POS_WEIGHT SWEEP
# ================================================================

SWEEP_RESULTS = []
RUN_HISTORY = []

for pos_weight in POS_WEIGHT_CANDIDATES:
    print('\n' + '=' * 72)
    print(f'BASE TRAINING — pos_weight={pos_weight:g}')
    print('=' * 72)

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    model = TinyUNet(base=8).to(device)

    bce = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight], dtype=torch.float32, device=device)
    )

    def loss_fn(logits, targets):
        return 0.3 * bce(logits, targets) + 0.7 * loss_tversky(logits, targets)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=4,
    )

    amp_enabled = device.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    best_val_f1 = -1.0
    best_epoch = -1
    stale = 0
    run_id = f'tiny_unet_b8_pw{int(pos_weight)}'
    checkpoint_path = OUTPUT_DIR / f'{run_id}_best.pt'

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []

        for images, masks in train_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=amp_enabled):
                logits = model(images)
                loss = loss_fn(logits, masks)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            train_losses.append(float(loss.item()))

        # The validation threshold is chosen on validation data, never test data.
        coarse_val, fine_val, best_val = choose_threshold(model, val_loader, loss_fn)
        scheduler.step(float(best_val['loss']))

        history_row = {
            'run_id': run_id,
            'epoch': epoch,
            'train_loss': float(np.mean(train_losses)),
            'val_loss': float(best_val['loss']),
            'val_f1': float(best_val['f1']),
            'val_iou': float(best_val['iou']),
            'val_precision': float(best_val['precision']),
            'val_recall': float(best_val['recall']),
            'val_frame_recall': float(best_val['frame_recall']),
            'val_frame_false_alarm_rate': float(best_val['frame_false_alarm_rate']),
            'val_threshold': float(best_val['threshold']),
            'lr': float(optimizer.param_groups[0]['lr']),
        }
        RUN_HISTORY.append(history_row)

        print(
            f"epoch={epoch:02d} train={history_row['train_loss']:.4f} "
            f"val_loss={history_row['val_loss']:.4f} "
            f"F1={history_row['val_f1']:.4f} "
            f"IoU={history_row['val_iou']:.4f} "
            f"P={history_row['val_precision']:.4f} "
            f"R={history_row['val_recall']:.4f} "
            f"thr={history_row['val_threshold']:.2f}"
        )

        if history_row['val_f1'] > best_val_f1:
            best_val_f1 = history_row['val_f1']
            best_epoch = epoch
            stale = 0
            torch.save(
                {
                    'model_state': model.state_dict(),
                    'base': 8,
                    'pos_weight': pos_weight,
                    'epoch': epoch,
                    'validation_f1': best_val_f1,
                    'validation_threshold': history_row['val_threshold'],
                    'normalization_min_k': NORM_MIN_K,
                    'normalization_max_k': NORM_MAX_K,
                },
                checkpoint_path,
            )
        else:
            stale += 1
            if stale >= PATIENCE:
                print('Early stopping.')
                break

    SWEEP_RESULTS.append({
        'run_id': run_id,
        'pos_weight': pos_weight,
        'parameters': sum(p.numel() for p in model.parameters()),
        'best_val_f1': best_val_f1,
        'best_epoch': best_epoch,
        'best_checkpoint': str(checkpoint_path),
    })

    del model, optimizer, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

history_df = pd.DataFrame(RUN_HISTORY)
sweep_df = pd.DataFrame(SWEEP_RESULTS)

history_df.to_csv(OUTPUT_DIR / 'base_training_history_v6_1.csv', index=False)
sweep_df.to_csv(OUTPUT_DIR / 'base_sweep_v6_1.csv', index=False)

display(sweep_df.sort_values('best_val_f1', ascending=False))


BASE TRAINING — pos_weight=10


/tmp/ipykernel_58/126890875.py:39: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
/tmp/ipykernel_58/126890875.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):


epoch=01 train=0.7193 val_loss=0.6737 F1=0.1900 IoU=0.1050 P=0.1402 R=0.2944 thr=0.49
epoch=02 train=0.6671 val_loss=0.6573 F1=0.2115 IoU=0.1183 P=0.1480 R=0.3710 thr=0.49
epoch=03 train=0.6598 val_loss=0.6544 F1=0.2352 IoU=0.1333 P=0.1743 R=0.3615 thr=0.49
epoch=04 train=0.6583 val_loss=0.6586 F1=0.3276 IoU=0.1959 P=0.3032 R=0.3564 thr=0.49
epoch=05 train=0.6556 val_loss=0.6503 F1=0.2844 IoU=0.1657 P=0.2103 R=0.4389 thr=0.49
epoch=06 train=0.6555 val_loss=0.6531 F1=0.3255 IoU=0.1944 P=0.2926 R=0.3668 thr=0.49
epoch=07 train=0.6524 val_loss=0.6511 F1=0.2832 IoU=0.1650 P=0.2343 R=0.3578 thr=0.49
epoch=08 train=0.6523 val_loss=0.6499 F1=0.2492 IoU=0.1424 P=0.1809 R=0.4007 thr=0.49
epoch=09 train=0.6539 val_loss=0.6509 F1=0.2959 IoU=0.1736 P=0.2471 R=0.3686 thr=0.49
Early stopping.

BASE TRAINING — pos_weight=15
epoch=01 train=0.7199 val_loss=0.6746 F1=0.1145 IoU=0.0608 P=0.0705 R=0.3050 thr=0.49
epoch=02 train=0.6689 val_loss=0.6613 F1=0.2447 IoU=0.1394 P=0.1948 R=0.3291 thr=0.49
epoch=0

,run_id,pos_weight,parameters,best_val_f1,best_epoch,best_checkpoint
2,tiny_unet_b8_pw25,25.0,121033,0.370594,10,/kaggle/working/africa_tiny_unet_v6_1/tiny_une...
1,tiny_unet_b8_pw15,15.0,121033,0.351387,5,/kaggle/working/africa_tiny_unet_v6_1/tiny_une...
0,tiny_unet_b8_pw10,10.0,121033,0.327619,4,/kaggle/working/africa_tiny_unet_v6_1/tiny_une...


In [44]:
# ================================================================
# RE-EVALUATE EXISTING BASE CHECKPOINTS
# ================================================================

CHECKPOINTS = {
    10.0: OUTPUT_DIR / "tiny_unet_b8_pw10_best.pt",
    15.0: OUTPUT_DIR / "tiny_unet_b8_pw15_best.pt",
    25.0: OUTPUT_DIR / "tiny_unet_b8_pw25_best.pt",
}

threshold_results = []

for pw, checkpoint_path in CHECKPOINTS.items():

    assert checkpoint_path.exists(), checkpoint_path

    model = TinyUNet(base=8).to(device)

    ckpt = torch.load(
        checkpoint_path,
        map_location=device,
    )

    model.load_state_dict(ckpt["model_state"])

    # The loss is only being reported here.
    # Threshold selection is based on F1.
    all_thresholds, best = choose_threshold_fast(
        model,
        val_eval_loader,
        loss_tversky,
    )

    print("\n" + "=" * 72)
    print(f"PW = {pw:g}")
    print("=" * 72)

    print(f"Best threshold: {best['threshold']:.2f}")
    print(f"Val F1:         {best['f1']:.4f}")
    print(f"Val IoU:        {best['iou']:.4f}")
    print(f"Val precision:  {best['precision']:.4f}")
    print(f"Val recall:     {best['recall']:.4f}")

    threshold_results.append({
        "pos_weight": pw,
        "threshold": float(best["threshold"]),
        "val_f1": float(best["f1"]),
        "val_iou": float(best["iou"]),
        "val_precision": float(best["precision"]),
        "val_recall": float(best["recall"]),
    })

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

threshold_results_df = pd.DataFrame(threshold_results)

print("\n========== RESULTS ==========")
display(
    threshold_results_df.sort_values(
        "val_f1",
        ascending=False,
    )
)


PW = 10
Best threshold: 0.95
Val F1:         0.3412
Val IoU:        0.2057
Val precision:  0.3649
Val recall:     0.3204


  6%|▌         | 1.91G/31.7G [35:00<9:05:23, 909kB/s]


KeyboardInterrupt: 

In [22]:
# ================================================================
# CELL 12 — SELECT BEST BASE MODEL + HELD-OUT TEST
# ================================================================

best_run = sweep_df.sort_values(
    ['best_val_f1', 'pos_weight'],
    ascending=[False, True],
).iloc[0]

BEST_CHECKPOINT = Path(best_run['best_checkpoint'])
BEST_POS_WEIGHT = float(best_run['pos_weight'])
assert BEST_CHECKPOINT.exists()

base_model = TinyUNet(base=8).to(device)
ckpt = torch.load(BEST_CHECKPOINT, map_location=device, weights_only=False)
base_model.load_state_dict(ckpt['model_state'])
base_model.eval()

bce_base = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([BEST_POS_WEIGHT], dtype=torch.float32, device=device)
)

def base_loss(logits, targets):
    return 0.3 * bce_base(logits, targets) + 0.7 * loss_tversky(logits, targets)

# Threshold is selected on validation only.
base_coarse, base_fine, best_base_val = choose_threshold(
    base_model,
    val_loader,
    base_loss,
)

BASE_FINAL_THRESHOLD = float(best_base_val['threshold'])
BASE_LOGIT_THRESHOLD = math.log(
    BASE_FINAL_THRESHOLD / (1 - BASE_FINAL_THRESHOLD)
)

print('Selected base threshold:', BASE_FINAL_THRESHOLD)
print_comparison('Base model — held-out test', {}) if False else None

base_test_metrics = evaluate(
    base_model,
    test_loader,
    base_loss,
    BASE_FINAL_THRESHOLD,
)
print_comparison('BASE MODEL — HELD-OUT AFRICA TEST', base_test_metrics)

# Also report the historical 0.25 operating point because the original ActiveFire
# repository uses 0.25 as the default fire threshold for its inference script.
base_test_at_025 = evaluate(
    base_model,
    test_loader,
    base_loss,
    0.25,
)
print('\nBASE MODEL — TEST AT FIXED THRESHOLD 0.25')
print_comparison('Fixed 0.25', base_test_at_025)

pd.DataFrame([{
    'experiment': 'base_africa',
    'threshold_selected_on_val': BASE_FINAL_THRESHOLD,
    **base_test_metrics,
}]).to_csv(
    OUTPUT_DIR / 'base_heldout_test_v6_1.csv',
    index=False,
)

Selected base threshold: 0.49
BASE MODEL — HELD-OUT AFRICA TEST
  F1:        0.4088
  IoU:       0.2569
  Precision: 0.4887
  Recall:    0.3514
  Frame recall: 0.3211
  False-alarm frame rate: 0.1341
  vs baseline 0.40: +0.0088 (+2.2%)

BASE MODEL — TEST AT FIXED THRESHOLD 0.25
Fixed 0.25
  F1:        0.4076
  IoU:       0.2560
  Precision: 0.4627
  Recall:    0.3643
  Frame recall: 0.3416
  False-alarm frame rate: 0.1605
  vs baseline 0.40: +0.0076 (+1.9%)


## 7. Manual annotation fine-tuning

The manual release is treated as an independent label source. Only manual patches whose stems exist in the African inventory are retained, so the fine-tuning experiment remains Africa-specific.

In [23]:
# ================================================================
# CELL 13 — DOWNLOAD MANUAL IMAGES + MANUAL ANNOTATIONS ONLY
# ================================================================

MANUAL_ROOT = SCRATCH_ROOT / 'manual_release'
MANUAL_ROOT.mkdir(parents=True, exist_ok=True)

MANUAL_FILE_IDS = {
    'landsat_images': '1uZnc65_GRFdAoavGoUKkJfeVQOUI8lVg',
    'manual_annotations': '1LdsX-rH5hy_82jfc1akO8p4n0_N8lRgf',
}

for name, file_id in MANUAL_FILE_IDS.items():
    archive = MANUAL_ROOT / f'{name}.zip'
    if not archive.exists():
        print(f'Downloading {name}...')
        gdown.download(id=file_id, output=str(archive), quiet=False)
    print(f'{name}: {archive.stat().st_size / 1024**3:.2f} GB')

for name in MANUAL_FILE_IDS:
    archive = MANUAL_ROOT / f'{name}.zip'
    target = MANUAL_ROOT / name
    if not target.exists():
        print(f'Extracting {name}...')
        with zipfile.ZipFile(archive, 'r') as zf:
            zf.extractall(target)

print('Manual release prepared under:', MANUAL_ROOT)

Downloading...
From (original): https://drive.google.com/uc?id=1uZnc65_GRFdAoavGoUKkJfeVQOUI8lVg
From (redirected): https://drive.google.com/uc?id=1uZnc65_GRFdAoavGoUKkJfeVQOUI8lVg&confirm=t&uuid=1ccf45c5-8793-45ca-a240-7658ddd2f4f7
To: /kaggle/tmp/fireedge/manual_release/landsat_images.zip
100%|██████████| 9.61G/9.61G [01:26<00:00, 112MB/s] 


landsat_images: 8.95 GB


Downloading...
From: https://drive.google.com/uc?id=1LdsX-rH5hy_82jfc1akO8p4n0_N8lRgf
To: /kaggle/tmp/fireedge/manual_release/manual_annotations.zip
100%|██████████| 82.9k/82.9k [00:00<00:00, 44.0MB/s]


manual_annotations: 0.00 GB
Extracting landsat_images...
Extracting manual_annotations...
Manual release prepared under: /kaggle/tmp/fireedge/manual_release


In [29]:
# ================================================================
# CELL 13D — DOWNLOAD ONLY THE MANUAL ANNOTATION ARCHIVE
# ================================================================

import gdown
import zipfile
import shutil
from pathlib import Path

MANUAL_ROOT = SCRATCH_ROOT / "manual_release"
MANUAL_ROOT.mkdir(parents=True, exist_ok=True)

MANUAL_ZIP = MANUAL_ROOT / "manual_annotations.zip"
MANUAL_EXTRACT = MANUAL_ROOT / "manual_annotations"

MANUAL_FILE_ID = "1LdsX-rH5hy_82jfc1akO8p4n0_N8lRgf"

# Remove any incomplete previous copy.
if MANUAL_ZIP.exists():
    MANUAL_ZIP.unlink()

if MANUAL_EXTRACT.exists():
    shutil.rmtree(MANUAL_EXTRACT)

print("Downloading manual annotations...")

gdown.download(
    id=MANUAL_FILE_ID,
    output=str(MANUAL_ZIP),
    quiet=False,
)

assert MANUAL_ZIP.exists(), "Manual annotation archive was not downloaded."
assert zipfile.is_zipfile(MANUAL_ZIP), "Downloaded file is not a valid ZIP."

size_kb = MANUAL_ZIP.stat().st_size / 1024
print(f"\n✅ Downloaded: {size_kb:.2f} KB")

# Inspect before extraction.
with zipfile.ZipFile(MANUAL_ZIP, "r") as zf:
    names = zf.namelist()

tifs = [
    n for n in names
    if n.lower().endswith((".tif", ".tiff"))
]

print(f"ZIP members: {len(names)}")
print(f"TIFF members: {len(tifs)}")

assert len(tifs) > 0, "No TIFF annotations found in archive."

print("\nFirst annotation files:")
for n in tifs[:10]:
    print(" ", n)

# Extract.
MANUAL_EXTRACT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(MANUAL_ZIP, "r") as zf:
    zf.extractall(MANUAL_EXTRACT)

manual_annotation_files = sorted(
    list(MANUAL_EXTRACT.rglob("*.tif")) +
    list(MANUAL_EXTRACT.rglob("*.tiff"))
)

print(f"\n✅ Extracted manual annotation TIFFs: {len(manual_annotation_files)}")

for p in manual_annotation_files[:10]:
    print(" ", p)

Downloading...
From: https://drive.google.com/uc?id=1LdsX-rH5hy_82jfc1akO8p4n0_N8lRgf
To: /kaggle/tmp/fireedge/manual_release/manual_annotations.zip

100%|██████████| 82.9k/82.9k [00:00<00:00, 50.2MB/s]


✅ Downloaded: 80.93 KB
ZIP members: 101
TIFF members: 100

First annotation files:
  manual_annotations_patches/LC08_L1TP_117016_20200926_20200926_01_RT_v1_p00586.tif
  manual_annotations_patches/LC08_L1TP_025033_20200921_20200921_01_RT_v1_p00677.tif
  manual_annotations_patches/LC08_L1TP_025033_20200921_20200921_01_RT_v1_p00601.tif
  manual_annotations_patches/LC08_L1TP_175074_20200916_20200916_01_RT_v1_p00137.tif
  manual_annotations_patches/LC08_L1TP_046031_20200908_20200908_01_RT_v1_p00604.tif
  manual_annotations_patches/LC08_L1TP_117016_20200926_20200926_01_RT_v1_p00213.tif
  manual_annotations_patches/LC08_L1TP_046031_20200908_20200908_01_RT_v1_p00846.tif
  manual_annotations_patches/LC08_L1TP_025033_20200921_20200921_01_RT_v1_p00481.tif
  manual_annotations_patches/LC08_L1TP_025033_20200921_20200921_01_RT_v1_p00275.tif
  manual_annotations_patches/LC08_L1TP_046021_20200908_20200908_01_RT_v1_p00843.tif

✅ Extracted manual annotation TIFFs: 100
  /kaggle/tmp/fireedge/manual_rele

In [30]:
# ================================================================
# FIND THE MANUAL ANNOTATION ARCHIVE
# ================================================================

from pathlib import Path

SEARCH_ROOT = Path("/kaggle/tmp/fireedge/manual_release")

matches = sorted(
    p for p in SEARCH_ROOT.rglob("*")
    if p.is_file()
    and (
        "manual_annotations" in p.name.lower()
        or "manual_annotation" in p.name.lower()
    )
)

print("Found matching files:", len(matches))

for p in matches:
    size_mb = p.stat().st_size / (1024 ** 2)
    print(f"{size_mb:8.2f} MB  {p}")

Found matching files: 1
    0.08 MB  /kaggle/tmp/fireedge/manual_release/manual_annotations.zip


In [33]:
# ================================================================
# CELL 14 — PAIR MANUAL ANNOTATIONS CORRECTLY + AFRICA FILTER
# ================================================================

from pathlib import Path
import re
import pandas as pd
import rasterio
from tqdm.auto import tqdm
import numpy as np

# ------------------------------------------------
# Directories
# ------------------------------------------------
LANDSAT_IMAGES_ROOT = MANUAL_ROOT / "landsat_images"
MANUAL_ANNOTATIONS_ROOT = MANUAL_ROOT / "manual_annotations"

assert LANDSAT_IMAGES_ROOT.exists()
assert MANUAL_ANNOTATIONS_ROOT.exists()

# ------------------------------------------------
# 1. Find manual masks and Landsat images
# ------------------------------------------------
manual_annotation_files = sorted(
    list(MANUAL_ANNOTATIONS_ROOT.rglob("*.tif")) +
    list(MANUAL_ANNOTATIONS_ROOT.rglob("*.tiff"))
)

manual_image_files = sorted(
    list(LANDSAT_IMAGES_ROOT.rglob("*.tif")) +
    list(LANDSAT_IMAGES_ROOT.rglob("*.tiff"))
)

print("Manual masks :", len(manual_annotation_files))
print("Landsat imgs :", len(manual_image_files))

assert len(manual_annotation_files) == 100
assert len(manual_image_files) == 9044

# ------------------------------------------------
# 2. Normalize filename for pairing
# ------------------------------------------------
def manual_pair_key(path):
    """
    Manual annotation files contain '_v1' before the patch number.
    Landsat image files do not.
    Example:

      manual: LC08_..._RT_v1_p00811.tif
      image : LC08_..._RT_p00811.tif

    Both become:
      LC08_..._RT_p00811
    """
    stem = Path(path).stem
    stem = re.sub(r"_v1(?=_p\d+$)", "", stem, flags=re.IGNORECASE)
    return stem.lower()

# ------------------------------------------------
# 3. Build image lookup
# ------------------------------------------------
images_by_key = {}

for p in manual_image_files:
    key = manual_pair_key(p)

    if key in images_by_key:
        raise RuntimeError(
            f"Duplicate normalized image key:\n"
            f"{key}\n"
            f"{images_by_key[key]}\n"
            f"{p}"
        )

    images_by_key[key] = p

# ------------------------------------------------
# 4. Pair the 100 manual annotations
# ------------------------------------------------
paired_records = []
unmatched_annotations = []

for mask_path in tqdm(
    manual_annotation_files,
    desc="Pairing manual annotations"
):
    key = manual_pair_key(mask_path)

    image_path = images_by_key.get(key)

    if image_path is None:
        unmatched_annotations.append(mask_path)
        continue

    # Validate mask
    with rasterio.open(mask_path) as src:
        mask = src.read(1)

    mask_binary = (mask > 0).astype(np.uint8)

    # Parent scene = filename without patch suffix
    parent_scene = re.sub(
        r"_p\d+$",
        "",
        Path(mask_path).stem
    )

    # Remove _v1 from scene name as well
    parent_scene = re.sub(
        r"_v1$",
        "",
        parent_scene,
        flags=re.IGNORECASE
    )

    paired_records.append({
        "stem": key,
        "image_path": str(image_path),
        "mask_path": str(mask_path),
        "fire_pixels": int(mask_binary.sum()),
        "parent_scene": parent_scene,
        "sample_type": "manual_annotation",
        "height": int(mask.shape[0]),
        "width": int(mask.shape[1]),
    })

manual_df_all = pd.DataFrame(paired_records)

print("\n========== MANUAL PAIRING ==========")
print("Manual annotations:   ", len(manual_annotation_files))
print("Successfully paired:  ", len(manual_df_all))
print("Unmatched annotations: ", len(unmatched_annotations))

if unmatched_annotations:
    print("\nFirst unmatched:")
    for p in unmatched_annotations[:10]:
        print(" ", p)

assert len(manual_df_all) == 100, (
    f"Expected 100 paired manual annotations, "
    f"got {len(manual_df_all)}"
)

# ------------------------------------------------
# 5. Check image/mask dimensions
# ------------------------------------------------
bad_shapes = manual_df_all[
    (manual_df_all["height"] != 256) |
    (manual_df_all["width"] != 256)
]

print("\nNon-256x256 masks:", len(bad_shapes))

assert len(bad_shapes) == 0

# ------------------------------------------------
# 6. Match against the SAME African dataset
# ------------------------------------------------
def normalize_inventory_stem(stem):
    stem = str(stem)
    stem = re.sub(
        r"_v1(?=_p\d+$)",
        "",
        stem,
        flags=re.IGNORECASE
    )
    return stem.lower()

africa_stems = {
    normalize_inventory_stem(s)
    for s in df_all["stem"]
}

manual_df = manual_df_all[
    manual_df_all["stem"].isin(africa_stems)
].copy()

# ------------------------------------------------
# 7. Report African manual subset
# ------------------------------------------------
print("\n========== AFRICA MANUAL SUBSET ==========")
print(
    "African manual patches:",
    len(manual_df)
)
print(
    "African manual scenes:",
    manual_df["parent_scene"].nunique()
)

print("\nManual fire-pixel statistics:")
print(
    manual_df["fire_pixels"].describe()
)

# ------------------------------------------------
# 8. Compare manual scenes with BASE split
# ------------------------------------------------
train_scenes = set(train_df["parent_scene"])
val_scenes   = set(val_df["parent_scene"])
test_scenes  = set(test_df["parent_scene"])

manual_scenes = set(manual_df["parent_scene"])

print("\n========== MANUAL / BASE SPLIT OVERLAP ==========")

print(
    "Manual scenes ∩ BASE TRAIN:",
    len(manual_scenes & train_scenes)
)

print(
    "Manual scenes ∩ BASE VAL:",
    len(manual_scenes & val_scenes)
)

print(
    "Manual scenes ∩ BASE TEST:",
    len(manual_scenes & test_scenes)
)

print("\nManual patches by BASE split:")

manual_train_mask = manual_df["parent_scene"].isin(train_scenes)
manual_val_mask   = manual_df["parent_scene"].isin(val_scenes)
manual_test_mask  = manual_df["parent_scene"].isin(test_scenes)

print(
    "  BASE TRAIN:",
    int(manual_train_mask.sum())
)

print(
    "  BASE VAL:  ",
    int(manual_val_mask.sum())
)

print(
    "  BASE TEST: ",
    int(manual_test_mask.sum())
)

print("\n✅ Manual annotations paired successfully.")
print("✅ Africa subset identified.")
print("✅ Base split overlap measured.")

Manual masks : 100
Landsat imgs : 9044


Pairing manual annotations:   0%|          | 0/100 [00:00<?, ?it/s]


========== MANUAL PAIRING ==========
Manual annotations:    100
Successfully paired:   100
Unmatched annotations:  0

Non-256x256 masks: 0

========== AFRICA MANUAL SUBSET ==========
African manual patches: 0
African manual scenes: 0

Manual fire-pixel statistics:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: fire_pixels, dtype: float64

========== MANUAL / BASE SPLIT OVERLAP ==========
Manual scenes ∩ BASE TRAIN: 0
Manual scenes ∩ BASE VAL: 0
Manual scenes ∩ BASE TEST: 0

Manual patches by BASE split:
  BASE TRAIN: 0
  BASE VAL:   0
  BASE TEST:  0

✅ Manual annotations paired successfully.
✅ Africa subset identified.
✅ Base split overlap measured.


In [34]:
# ================================================================
# CELL 14B — CHECK MANUAL SCENE OVERLAP WITH AFRICA DATASET
# ================================================================

# manual_df_all already contains the 100 correctly paired manual patches.

africa_train_scenes = set(train_df["parent_scene"])
africa_val_scenes   = set(val_df["parent_scene"])
africa_test_scenes  = set(test_df["parent_scene"])
africa_all_scenes   = (
    africa_train_scenes
    | africa_val_scenes
    | africa_test_scenes
)

manual_scenes = set(manual_df_all["parent_scene"])

print("========== MANUAL DATASET ==========")
print("Manual patches:", len(manual_df_all))
print("Manual scenes :", len(manual_scenes))

print("\n========== SCENE OVERLAP ==========")
print(
    "Manual ∩ Africa TRAIN:",
    len(manual_scenes & africa_train_scenes)
)
print(
    "Manual ∩ Africa VAL:",
    len(manual_scenes & africa_val_scenes)
)
print(
    "Manual ∩ Africa TEST:",
    len(manual_scenes & africa_test_scenes)
)
print(
    "Manual ∩ Africa ALL:",
    len(manual_scenes & africa_all_scenes)
)

print("\nManual patches whose scene is in Africa dataset:")

manual_scene_overlap = manual_df_all["parent_scene"].isin(
    africa_all_scenes
)

print(
    "  Overlapping patches:",
    int(manual_scene_overlap.sum())
)

print(
    "  Non-overlapping patches:",
    int((~manual_scene_overlap).sum())
)

print("\nManual scenes:")
for s in sorted(manual_scenes):
    print(" ", s)

========== MANUAL DATASET ==========
Manual patches: 100
Manual scenes : 10

========== SCENE OVERLAP ==========
Manual ∩ Africa TRAIN: 0
Manual ∩ Africa VAL: 0
Manual ∩ Africa TEST: 0
Manual ∩ Africa ALL: 0

Manual patches whose scene is in Africa dataset:
  Overlapping patches: 0
  Non-overlapping patches: 100

Manual scenes:
  LC08_L1GT_226074_20200921_20200921_01_RT
  LC08_L1TP_025033_20200921_20200921_01_RT
  LC08_L1TP_046021_20200908_20200908_01_RT
  LC08_L1TP_046031_20200908_20200908_01_RT
  LC08_L1TP_105069_20200922_20200922_01_RT
  LC08_L1TP_117016_20200926_20200926_01_RT
  LC08_L1TP_134039_20200901_20200901_01_RT
  LC08_L1TP_157012_20200918_20200918_01_RT
  LC08_L1TP_175074_20200916_20200916_01_RT
  LC08_L1TP_177044_20200914_20200914_01_RT


In [35]:
# ================================================================
# CELL 15 — SCENE-LEVEL MANUAL FINE-TUNING SPLIT
# ================================================================

import numpy as np
import pandas as pd

assert len(manual_df_all) == 100
assert manual_df_all["parent_scene"].nunique() == 10

SEED_MANUAL = 42

manual_scenes = np.array(
    sorted(manual_df_all["parent_scene"].unique())
)

rng = np.random.default_rng(SEED_MANUAL)
rng.shuffle(manual_scenes)

# 6 train scenes / 2 validation scenes / 2 test scenes
manual_train_scenes = set(manual_scenes[:6])
manual_val_scenes   = set(manual_scenes[6:8])
manual_test_scenes  = set(manual_scenes[8:10])

manual_train_df = manual_df_all[
    manual_df_all["parent_scene"].isin(manual_train_scenes)
].copy()

manual_val_df = manual_df_all[
    manual_df_all["parent_scene"].isin(manual_val_scenes)
].copy()

manual_test_df = manual_df_all[
    manual_df_all["parent_scene"].isin(manual_test_scenes)
].copy()

print("========== MANUAL SCENE SPLIT ==========")

print(
    f"Train: {len(manual_train_df)} patches / "
    f"{manual_train_df['parent_scene'].nunique()} scenes"
)

print(
    f"Val:   {len(manual_val_df)} patches / "
    f"{manual_val_df['parent_scene'].nunique()} scenes"
)

print(
    f"Test:  {len(manual_test_df)} patches / "
    f"{manual_test_df['parent_scene'].nunique()} scenes"
)

print("\nPatch totals:")
print(
    len(manual_train_df)
    + len(manual_val_df)
    + len(manual_test_df)
)

print("\nScene assignments:")

for scene in manual_scenes:
    if scene in manual_train_scenes:
        split = "TRAIN"
    elif scene in manual_val_scenes:
        split = "VAL"
    else:
        split = "TEST"

    n = int(
        (manual_df_all["parent_scene"] == scene).sum()
    )

    print(f"{split:5s} | {n:3d} patches | {scene}")

# ------------------------------------------------
# Leakage checks
# ------------------------------------------------
assert manual_train_scenes.isdisjoint(manual_val_scenes)
assert manual_train_scenes.isdisjoint(manual_test_scenes)
assert manual_val_scenes.isdisjoint(manual_test_scenes)

# Manual data must remain separate from African base scenes.
assert manual_train_scenes.isdisjoint(set(train_df["parent_scene"]))
assert manual_val_scenes.isdisjoint(set(val_df["parent_scene"]))
assert manual_test_scenes.isdisjoint(set(test_df["parent_scene"]))

print("\n✅ No manual train/val/test scene overlap.")
print("✅ No overlap with African base train/val/test scenes.")

========== MANUAL SCENE SPLIT ==========
Train: 45 patches / 6 scenes
Val:   8 patches / 2 scenes
Test:  47 patches / 2 scenes

Patch totals:
100

Scene assignments:
TRAIN |  18 patches | LC08_L1TP_117016_20200926_20200926_01_RT
TRAIN |   1 patches | LC08_L1TP_134039_20200901_20200901_01_RT
TRAIN |   2 patches | LC08_L1GT_226074_20200921_20200921_01_RT
TRAIN |   5 patches | LC08_L1TP_157012_20200918_20200918_01_RT
TRAIN |  15 patches | LC08_L1TP_046031_20200908_20200908_01_RT
TRAIN |   4 patches | LC08_L1TP_046021_20200908_20200908_01_RT
VAL   |   7 patches | LC08_L1TP_105069_20200922_20200922_01_RT
VAL   |   1 patches | LC08_L1TP_177044_20200914_20200914_01_RT
TEST  |  38 patches | LC08_L1TP_025033_20200921_20200921_01_RT
TEST  |   9 patches | LC08_L1TP_175074_20200916_20200916_01_RT

✅ No manual train/val/test scene overlap.
✅ No overlap with African base train/val/test scenes.


In [36]:
# ================================================================
# CELL 16 — MANUAL DATALOADERS
# ================================================================

manual_train_ds = ActiveFireDataset(
    manual_train_df,
    channels=(8,),
    norm_method='fixed',
    calibrate_temperature=True,
    degrader=None,
    augment=True,
    random_seed=SEED,
)
manual_val_ds = ActiveFireDataset(
    manual_val_df,
    channels=(8,),
    norm_method='fixed',
    calibrate_temperature=True,
    degrader=None,
    augment=False,
)
manual_test_ds = ActiveFireDataset(
    manual_test_df,
    channels=(8,),
    norm_method='fixed',
    calibrate_temperature=True,
    degrader=None,
    augment=False,
)

manual_train_loader = DataLoader(
    manual_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)
manual_val_loader = DataLoader(
    manual_val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)
manual_test_loader = DataLoader(
    manual_test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

mx, my = next(iter(manual_train_loader))
print('Manual input:', tuple(mx.shape), 'range:', float(mx.min()), float(mx.max()))
print('Manual mask:', tuple(my.shape), 'values:', torch.unique(my))
print("\nManual dataset sizes:")
print("Train:", len(manual_train_ds))
print("Val:  ", len(manual_val_ds))
print("Test: ", len(manual_test_ds))

assert len(manual_train_ds) > 0
assert len(manual_val_ds) > 0
assert len(manual_test_ds) > 0

mx, my = next(iter(manual_train_loader))
print('\nManual input:', tuple(mx.shape), 'range:', float(mx.min()), float(mx.max()))
print('Manual mask:', tuple(my.shape), 'values:', torch.unique(my))

Manual input: (8, 1, 256, 256) range: 0.0 0.6535379886627197
Manual mask: (8, 1, 256, 256) values: tensor([0., 1.])

Manual dataset sizes:
Train: 45
Val:   8
Test:  47

Manual input: (8, 1, 256, 256) range: 0.0 0.6535379886627197
Manual mask: (8, 1, 256, 256) values: tensor([0., 1.])


In [37]:
# ================================================================
# CELL 17 — MANUAL FINE-TUNING
# ================================================================

# Start from the best Africa model.
ft_model = TinyUNet(base=8).to(device)
base_ckpt = torch.load(BEST_CHECKPOINT, map_location=device, weights_only=False)
ft_model.load_state_dict(base_ckpt['model_state'])

manual_fire_fraction = (
    manual_train_df['fire_pixels'].sum()
    / max(len(manual_train_df) * 256 * 256, 1)
)
manual_pos_weight = min(
    25.0,
    (1 - manual_fire_fraction) / max(manual_fire_fraction, 1e-9),
)

bce_manual = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([manual_pos_weight], dtype=torch.float32, device=device)
)

def manual_loss(logits, targets):
    return 0.3 * bce_manual(logits, targets) + 0.7 * loss_tversky(logits, targets)

# Conservative fine-tuning: freeze early encoder for 3 epochs, then unfreeze.
for module in (ft_model.e1, ft_model.e2, ft_model.e3):
    for p in module.parameters():
        p.requires_grad = False

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, ft_model.parameters()),
    lr=1e-4,
    weight_decay=1e-4,
)

FT_EPOCHS = 15
FT_PATIENCE = 5
best_ft_f1 = -1.0
stale = 0
ft_checkpoint_path = OUTPUT_DIR / 'best_tiny_unet_manual_v6_1.pt'
ft_history = []

amp_enabled = device.type == 'cuda'
scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

for epoch in range(1, FT_EPOCHS + 1):
    if epoch == 4:
        for module in (ft_model.e1, ft_model.e2, ft_model.e3):
            for p in module.parameters():
                p.requires_grad = True
        optimizer = torch.optim.AdamW(
            ft_model.parameters(),
            lr=5e-5,
            weight_decay=1e-4,
        )
        print('Encoder unfrozen; LR reduced to 5e-5.')

    ft_model.train()
    losses = []

    for images, masks in manual_train_loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=amp_enabled):
            logits = ft_model(images)
            loss = manual_loss(logits, masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        losses.append(float(loss.item()))

    _, _, best_val = choose_threshold(
        ft_model,
        manual_val_loader,
        manual_loss,
    )

    row = {
        'epoch': epoch,
        'train_loss': float(np.mean(losses)),
        'val_loss': float(best_val['loss']),
        'val_f1': float(best_val['f1']),
        'val_iou': float(best_val['iou']),
        'val_precision': float(best_val['precision']),
        'val_recall': float(best_val['recall']),
        'val_threshold': float(best_val['threshold']),
    }
    ft_history.append(row)

    print(
        f"epoch={epoch:02d} train={row['train_loss']:.4f} "
        f"val={row['val_loss']:.4f} F1={row['val_f1']:.4f} "
        f"IoU={row['val_iou']:.4f} P={row['val_precision']:.4f} "
        f"R={row['val_recall']:.4f} thr={row['val_threshold']:.2f}"
    )

    if row['val_f1'] > best_ft_f1:
        best_ft_f1 = row['val_f1']
        stale = 0
        torch.save(
            {
                'model_state': ft_model.state_dict(),
                'base': 8,
                'pos_weight': manual_pos_weight,
                'epoch': epoch,
                'validation_f1': best_ft_f1,
                'validation_threshold': row['val_threshold'],
                'normalization_min_k': NORM_MIN_K,
                'normalization_max_k': NORM_MAX_K,
            },
            ft_checkpoint_path,
        )
    else:
        stale += 1
        if stale >= FT_PATIENCE:
            print('Manual fine-tuning early stopped.')
            break

pd.DataFrame(ft_history).to_csv(
    OUTPUT_DIR / 'manual_finetune_history_v6_1.csv',
    index=False,
)

print('Best manual-validation F1:', best_ft_f1)
print('Checkpoint:', ft_checkpoint_path)

/tmp/ipykernel_58/3818584275.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
/tmp/ipykernel_58/3818584275.py:67: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):


epoch=01 train=2.2363 val=0.5503 F1=0.5734 IoU=0.4020 P=0.5376 R=0.6144 thr=0.48
epoch=02 train=2.2490 val=0.5617 F1=0.5770 IoU=0.4055 P=0.5734 R=0.5807 thr=0.49
epoch=03 train=2.3593 val=0.5690 F1=0.5761 IoU=0.4046 P=0.5833 R=0.5691 thr=0.45
Encoder unfrozen; LR reduced to 5e-5.
epoch=04 train=2.3858 val=0.5677 F1=0.5759 IoU=0.4044 P=0.5769 R=0.5749 thr=0.41
epoch=05 train=2.1232 val=0.5669 F1=0.5754 IoU=0.4039 P=0.5747 R=0.5761 thr=0.46
epoch=06 train=2.3138 val=0.5650 F1=0.5719 IoU=0.4005 P=0.5645 R=0.5796 thr=0.48
epoch=07 train=2.3733 val=0.5647 F1=0.5726 IoU=0.4011 P=0.5636 R=0.5819 thr=0.49
Manual fine-tuning early stopped.
Best manual-validation F1: 0.5770340450083226
Checkpoint: /kaggle/working/africa_tiny_unet_v6_1/best_tiny_unet_manual_v6_1.pt


In [38]:
# ================================================================
# CELL 17 — HELD-OUT MANUAL TEST
# ================================================================

ft_ckpt = torch.load(ft_checkpoint_path, map_location=device, weights_only=False)
final_model = TinyUNet(base=8).to(device)
final_model.load_state_dict(ft_ckpt['model_state'])
final_model.eval()

_, _, best_manual_val = choose_threshold(
    final_model,
    manual_val_loader,
    manual_loss,
)

MANUAL_FINAL_THRESHOLD = float(best_manual_val['threshold'])
MANUAL_LOGIT_THRESHOLD = math.log(
    MANUAL_FINAL_THRESHOLD / (1 - MANUAL_FINAL_THRESHOLD)
)

manual_test_metrics = evaluate(
    final_model,
    manual_test_loader,
    manual_loss,
    MANUAL_FINAL_THRESHOLD,
)

print_comparison(
    'MANUAL-FINE-TUNED MODEL — HELD-OUT MANUAL TEST',
    manual_test_metrics,
)

comparison = pd.DataFrame([
    {
        'experiment': 'Africa base',
        'threshold': BASE_FINAL_THRESHOLD,
        **base_test_metrics,
    },
    {
        'experiment': 'Africa base @ 0.25',
        'threshold': 0.25,
        **base_test_at_025,
    },
    {
        'experiment': 'Manual fine-tuned',
        'threshold': MANUAL_FINAL_THRESHOLD,
        **manual_test_metrics,
    },
])

display(
    comparison[
        [
            'experiment',
            'threshold',
            'f1',
            'iou',
            'precision',
            'recall',
            'frame_recall',
            'frame_false_alarm_rate',
        ]
    ]
)

comparison.to_csv(
    OUTPUT_DIR / 'paper_ready_model_comparison_v6_1.csv',
    index=False,
)

MANUAL-FINE-TUNED MODEL — HELD-OUT MANUAL TEST
  F1:        0.4076
  IoU:       0.2560
  Precision: 0.8569
  Recall:    0.2674
  Frame recall: 0.2340
  False-alarm frame rate: 0.0000
  vs baseline 0.40: +0.0076 (+1.9%)


,experiment,threshold,f1,iou,precision,recall,frame_recall,frame_false_alarm_rate
0,Africa base,0.49,0.408827,0.256934,0.488739,0.351375,0.321085,0.134132
1,Africa base @ 0.25,0.25,0.407630,0.255990,0.462689,0.364282,0.341612,0.160479
2,Manual fine-tuned,0.49,0.407638,0.255996,0.856909,0.267428,0.234043,0.000000


In [39]:
# ================================================================
# CELL 17B — GATE 1 -> PREPROCESSING -> CNN -> GATE 2 SMOKE TEST
# ================================================================

# This is a SOFTWARE INTEGRATION CHECK on Landsat B10.
# It is NOT a physical validation of the final uncooled LWIR microbolometer.
# In particular, the real flight NUC stage is not available in this notebook.

import time
from fireedge.validation.base_validator import ValidatorConfig
from fireedge.validation.ground_gate_validator import GroundGateValidator
from fireedge.validation.flight_gate_validator import FlightGateValidator
from fireedge.validation.output_validator import OutputValidator
from fireedge.preprocessing.calibration import dn_to_temperature
from fireedge.preprocessing.normalization import fixed_window_norm

# Ground Gate 1 is the dataset-curation mode and expects multi-band input.
ground_gate = GroundGateValidator(
    ValidatorConfig(cross_channel_confirm=True)
)

# Flight Gate 1 is the onboard single-channel mode.
flight_gate = FlightGateValidator()
output_gate = OutputValidator()

# First exercise Ground Gate 1 on a small B10+B11 sample.
ground_rows = train_df.head(min(16, len(train_df)))
ground_status = []
for _, row in ground_rows.iterrows():
    with rasterio.open(row['image_path']) as src:
        raw_b10_b11 = src.read([9, 10]).astype(np.float32)  # Landsat B10/B11
    rep = ground_gate.validate(raw_b10_b11)
    ground_status.append(rep.status.value)

print('Ground Gate 1 sample audit:', pd.Series(ground_status).value_counts().to_dict())

# Find a Landsat test frame that passes the software Flight Gate 1 so the
# downstream model + Gate 2 path can also be exercised.
selected = None
for _, row in test_df.head(min(32, len(test_df))).iterrows():
    with rasterio.open(row['image_path']) as src:
        raw_b10 = src.read(9).astype(np.float32)  # Rasterio band 9 = Landsat B10
    rep = flight_gate.validate(raw_b10)
    if rep.ok:
        selected = (row, raw_b10, rep)
        break

PIPELINE_NUC_INCLUDED = False

if selected is None:
    print('⚠️ No sampled Landsat B10 test frame passed Flight Gate 1.')
    print('No downstream end-to-end smoke timing was produced.')
    print('This is a software-path result only; it does not establish flight-gate performance.')
else:
    selected_row, raw_b10, gate1_report = selected

    t0 = time.perf_counter()
    gate1_report = flight_gate.validate(raw_b10)
    gate1_ms = (time.perf_counter() - t0) * 1000.0

    # IMPORTANT: NUC is deliberately explicit here as a missing stage.
    # Do not present the resulting total as a complete flight-pipeline timing.
    t0 = time.perf_counter()
    temp_k = dn_to_temperature(raw_b10, band='B10', unit='kelvin')
    normalized = fixed_window_norm(
        temp_k,
        vmin_k=NORM_MIN_K,
        vmax_k=NORM_MAX_K,
    ).astype(np.float32)
    prep_report = output_gate.validate_preprocessing(normalized)
    preprocessing_ms = (time.perf_counter() - t0) * 1000.0

    if not gate1_report.ok or not prep_report.ok:
        print('Gate path stopped before CNN inference.')
        print('Gate 1:', gate1_report.status.value, gate1_report.reasons)
        print('Preprocessing:', prep_report.status.value, prep_report.reasons)
    else:
        x = torch.from_numpy(normalized[None, None]).to(device)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            probability = torch.sigmoid(final_model(x))[0, 0].detach().cpu().numpy()
        if device.type == 'cuda':
            torch.cuda.synchronize()
        model_ms = (time.perf_counter() - t0) * 1000.0

        pred_mask = (probability >= MANUAL_FINAL_THRESHOLD).astype(np.uint8)

        t0 = time.perf_counter()
        gate2_report = output_gate.validate_prediction(
            pred_mask,
            raw_thermal=raw_b10,
            confidence_map=probability,
        )
        gate2_ms = (time.perf_counter() - t0) * 1000.0

        print('\n===== SOFTWARE PIPELINE SMOKE RESULT =====')
        print('Test frame:', selected_row['stem'])
        print('Gate 1:', gate1_report.status.value, gate1_report.reasons)
        print(f'Gate 1 time: {gate1_ms:.3f} ms')
        print('NUC included:', PIPELINE_NUC_INCLUDED)
        print(f'Calibration + normalization time: {preprocessing_ms:.3f} ms')
        print(f'CNN inference time ({device.type}): {model_ms:.3f} ms')
        print('Gate 2:', gate2_report.status.value, gate2_report.reasons)
        print(f'Gate 2 time: {gate2_ms:.3f} ms')
        print(
            f'Partial software chain total (excluding NUC): '
            f'{gate1_ms + preprocessing_ms + model_ms + gate2_ms:.3f} ms'
        )
        print('⚠️ This is a Landsat/Kaggle software smoke test, not a Cortex-A72 flight benchmark.')

        gate_smoke_df = pd.DataFrame([{
            'frame_stem': selected_row['stem'],
            'gate1_status': gate1_report.status.value,
            'gate1_ms': gate1_ms,
            'nuc_included': PIPELINE_NUC_INCLUDED,
            'preprocessing_ms': preprocessing_ms,
            'cnn_device': device.type,
            'cnn_ms': model_ms,
            'gate2_status': gate2_report.status.value,
            'gate2_ms': gate2_ms,
            'partial_chain_ms_excluding_nuc': gate1_ms + preprocessing_ms + model_ms + gate2_ms,
            'note': 'Landsat B10 software smoke test; not target-hardware or flight-sensor validation.',
        }])
        display(gate_smoke_df)
        gate_smoke_df.to_csv(
            OUTPUT_DIR / 'gate_pipeline_smoke_test_v6_1.csv', index=False
        )

Ground Gate 1 sample audit: {'PASS_WITH_FLAGS': 14, 'REJECT': 2}

===== SOFTWARE PIPELINE SMOKE RESULT =====
Test frame: LC08_L1GT_166058_20200816_20200816_01_RT_p00738
Gate 1: PASS_WITH_FLAGS ['ch0:STRIPING']
Gate 1 time: 12.657 ms
NUC included: False
Calibration + normalization time: 0.937 ms
CNN inference time (cuda): 4.404 ms
Gate 2: PASS []
Gate 2 time: 0.158 ms
Partial software chain total (excluding NUC): 18.155 ms
⚠️ This is a Landsat/Kaggle software smoke test, not a Cortex-A72 flight benchmark.


,frame_stem,gate1_status,gate1_ms,nuc_included,preprocessing_ms,cnn_device,cnn_ms,gate2_status,gate2_ms,partial_chain_ms_excluding_nuc,note
0,LC08_L1GT_166058_20200816_20200816_01_RT_p00738,PASS_WITH_FLAGS,12.656502,False,0.936583,cuda,4.404469,PASS,0.157863,18.155417,Landsat B10 software smoke test; not target-ha...


## 8. ONNX + static INT8 export

The model being exported is the **manual-fine-tuned model**, with the validation-selected threshold stored separately. The threshold is not embedded into the neural network weights.

In [53]:
# ================================================================
# CELL 18 — EXPORT BASE AFRICA FP32 ONNX + STATIC INT8
# ================================================================

!pip -q install onnx onnxruntime onnxsim onnxscript

import onnx
from onnxsim import simplify
from onnxruntime.quantization import (
    CalibrationDataReader,
    quantize_static,
    QuantType,
    QuantFormat,
)

# ------------------------------------------------
# 1. Load the BEST AFRICA BASE CHECKPOINT
# ------------------------------------------------
export_model = TinyUNet(base=8).cpu()

base_ckpt = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

export_model.load_state_dict(base_ckpt["model_state"])
export_model.eval()

print(
    "Exporting BASE AFRICA model:",
    BEST_CHECKPOINT,
)

print(
    "Parameters:",
    sum(p.numel() for p in export_model.parameters())
)

# ------------------------------------------------
# 2. Export FP32 ONNX
# ------------------------------------------------
example = torch.zeros(
    1, 1, 256, 256,
    dtype=torch.float32,
)

onnx_fp32 = OUTPUT_DIR / "tiny_unet_africa_base_fp32_v6_1.onnx"

torch.onnx.export(
    export_model,
    example,
    str(onnx_fp32),
    input_names=["thermal_b10"],
    output_names=["fire_logits"],
    opset_version=18,
    do_constant_folding=True,
)

print("FP32 ONNX export completed:", onnx_fp32)

# ------------------------------------------------
# 3. Validate the exported ONNX model
# ------------------------------------------------
onnx_model = onnx.load(str(onnx_fp32))
onnx.checker.check_model(onnx_model)

print("ONNX checker: PASS")

# ------------------------------------------------
# 4. Simplify ONNX
# ------------------------------------------------
simplified_model, ok = simplify(onnx_model)
assert ok, "ONNX simplification failed."

onnx_sim = (
    OUTPUT_DIR /
    "tiny_unet_africa_base_fp32_simplified_v6_1.onnx"
)

onnx.save(
    simplified_model,
    str(onnx_sim),
)

print("Simplified ONNX saved:", onnx_sim)

# ------------------------------------------------
# 5. Dedicated calibration loader
#    Use VALIDATION data, no multiprocessing.
# ------------------------------------------------
calibration_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

print(
    "Calibration loader prepared:",
    len(calibration_loader),
    "batches",
)

# ------------------------------------------------
# 6. ONNX Runtime calibration reader
# ------------------------------------------------
class CalibrationReader(CalibrationDataReader):

    def __init__(self, loader, max_batches=32):
        self.items = []

        for batch_index, (images, _) in enumerate(loader):

            # images shape:
            # [B, 1, 256, 256]

            for image in images:

                # image shape:
                # [1, 256, 256]

                # Add batch dimension:
                # [1, 1, 256, 256]

                sample = image.unsqueeze(0).numpy().astype(
                    np.float32
                )

                assert sample.shape == (1, 1, 256, 256), (
                    f"Bad calibration shape: {sample.shape}"
                )

                self.items.append({
                    "thermal_b10": sample
                })

            if batch_index + 1 >= max_batches:
                break

        self.iterator = iter(self.items)

    def get_next(self):
        return next(self.iterator, None)


calibration_reader = CalibrationReader(
    calibration_loader,
    max_batches=32,
)

print(
    "Calibration samples:",
    len(calibration_reader.items)
)

# ------------------------------------------------
# 7. Static INT8 quantization
# ------------------------------------------------
onnx_int8 = (
    OUTPUT_DIR /
    "tiny_unet_africa_base_static_int8_v6_1.onnx"
)

quantize_static(
    model_input=str(onnx_sim),
    model_output=str(onnx_int8),
    calibration_data_reader=calibration_reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QInt8,
    weight_type=QuantType.QInt8,
    per_channel=True,
)

# ------------------------------------------------
# 8. Report
# ------------------------------------------------
def size_mb(path):
    return Path(path).stat().st_size / 1024**2


print()
print("===== BASE AFRICA ONNX EXPORT RESULT =====")
print("Parameters      :", sum(
    p.numel() for p in export_model.parameters()
))
print("FP32 ONNX       :", round(size_mb(onnx_sim), 4), "MB")
print("INT8 ONNX       :", round(size_mb(onnx_int8), 4), "MB")
print("FP32 exists     :", onnx_sim.exists())
print("INT8 exists     :", onnx_int8.exists())
print("Base threshold  :", BASE_FINAL_THRESHOLD)
print("Input contract  : [1, 1, 256, 256]")
print("Input name      : thermal_b10")
print("Output name     : fire_logits")

Exporting BASE AFRICA model: /kaggle/working/africa_tiny_unet_v6_1/tiny_unet_b8_pw25_best.pt
Parameters: 121033
[torch.onnx] Obtain model graph for `TinyUNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `TinyUNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
FP32 ONNX export completed: /kaggle/working/africa_tiny_unet_v6_1/tiny_unet_africa_base_fp32_v6_1.onnx
ONNX checker: PASS
Simplified ONNX saved: /kaggle/working/africa_tiny_unet_v6_1/tiny_unet_africa_base_fp32_simplified_v6_1.onnx
Calibration loader prepared: 503 batches


Calibration samples: 256



===== BASE AFRICA ONNX EXPORT RESULT =====
Parameters      : 121033
FP32 ONNX       : 0.4677 MB
INT8 ONNX       : 0.1467 MB
FP32 exists     : True
INT8 exists     : True
Base threshold  : 0.49
Input contract  : [1, 1, 256, 256]
Input name      : thermal_b10
Output name     : fire_logits


In [56]:
# ================================================================
# CELL 19 — PYTORCH vs ONNX / INT8 NUMERICAL CHECK
# ================================================================

import onnxruntime as ort
import numpy as np
import torch

# ------------------------------------------------
# 1. Load BASE AFRICA model
# ------------------------------------------------
check_model = TinyUNet(base=8).to(device)

base_ckpt = torch.load(
    BEST_CHECKPOINT,
    map_location=device,
    weights_only=False,
)

check_model.load_state_dict(base_ckpt["model_state"])
check_model.eval()

print("Checking model:", BEST_CHECKPOINT)
print(
    "Parameters:",
    sum(p.numel() for p in check_model.parameters())
)

# ------------------------------------------------
# 2. Safe loader: single-worker
# ------------------------------------------------
check_loader = DataLoader(
    test_ds,
    batch_size=1,          # IMPORTANT: ONNX was exported for batch=1
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

check_images, check_masks = next(iter(check_loader))

print("PyTorch input shape:", tuple(check_images.shape))
print("PyTorch input dtype:", check_images.dtype)

assert check_images.shape == (1, 1, 256, 256)

check_np = check_images.numpy().astype(np.float32)

# ------------------------------------------------
# 3. PyTorch inference
# ------------------------------------------------
with torch.inference_mode():
    pytorch_logits = check_model(
        check_images.to(device)
    )

pytorch_np = (
    pytorch_logits
    .detach()
    .cpu()
    .numpy()
    .astype(np.float32)
)

print("PyTorch output shape:", pytorch_np.shape)

# ------------------------------------------------
# 4. FP32 ONNX
# ------------------------------------------------
fp32_session = ort.InferenceSession(
    str(onnx_sim),
    providers=["CPUExecutionProvider"],
)

fp32_input = fp32_session.get_inputs()[0]
fp32_output = fp32_session.get_outputs()[0]

print("ONNX expected input shape:", fp32_input.shape)
print("ONNX input name:", fp32_input.name)

onnx_fp32_output = fp32_session.run(
    [fp32_output.name],
    {fp32_input.name: check_np},
)[0]

print("ONNX FP32 output shape:", onnx_fp32_output.shape)

# ------------------------------------------------
# 5. PyTorch vs FP32 ONNX
# ------------------------------------------------
fp32_abs_diff = np.abs(
    pytorch_np - onnx_fp32_output
)

print()
print("===== PYTORCH vs ONNX FP32 =====")
print(
    "Max absolute difference :",
    float(fp32_abs_diff.max())
)
print(
    "Mean absolute difference:",
    float(fp32_abs_diff.mean())
)

assert np.allclose(
    pytorch_np,
    onnx_fp32_output,
    rtol=1e-3,
    atol=1e-4,
)

print("FP32 numerical check: PASS")

# ------------------------------------------------
# 6. INT8 ONNX
# ------------------------------------------------
int8_session = ort.InferenceSession(
    str(onnx_int8),
    providers=["CPUExecutionProvider"],
)

int8_input = int8_session.get_inputs()[0]
int8_output = int8_session.get_outputs()[0]

onnx_int8_output = int8_session.run(
    [int8_output.name],
    {int8_input.name: check_np},
)[0]

print("ONNX INT8 output shape:", onnx_int8_output.shape)

# ------------------------------------------------
# 7. INT8 comparison
# ------------------------------------------------
int8_abs_diff = np.abs(
    onnx_fp32_output - onnx_int8_output
)

p_fp32 = 1.0 / (1.0 + np.exp(-onnx_fp32_output))
p_int8 = 1.0 / (1.0 + np.exp(-onnx_int8_output))

prob_abs_diff = np.abs(p_fp32 - p_int8)

print()
print("===== ONNX FP32 vs INT8 =====")
print(
    "Max logit difference       :",
    float(int8_abs_diff.max())
)
print(
    "Mean logit difference      :",
    float(int8_abs_diff.mean())
)
print(
    "Max probability difference :",
    float(prob_abs_diff.max())
)
print(
    "Mean probability difference:",
    float(prob_abs_diff.mean())
)

assert np.isfinite(onnx_int8_output).all()
assert np.isfinite(p_int8).all()

print("INT8 numerical sanity check: PASS")

# ------------------------------------------------
# 8. Final contract check
# ------------------------------------------------
assert check_np.shape == (1, 1, 256, 256)
assert pytorch_np.shape == (1, 1, 256, 256)
assert onnx_fp32_output.shape == (1, 1, 256, 256)
assert onnx_int8_output.shape == (1, 1, 256, 256)

print()
print("===== CELL 19 RESULT =====")
print("PyTorch vs FP32 ONNX : PASS")
print("FP32 ONNX             : PASS")
print("INT8 ONNX             : PASS")
print("Model                 : BASE AFRICA")
print("Parameters            :",
      sum(p.numel() for p in check_model.parameters()))
print("Input                 : [1, 1, 256, 256]")
print("Threshold             :", BASE_FINAL_THRESHOLD)

Checking model: /kaggle/working/africa_tiny_unet_v6_1/tiny_unet_b8_pw25_best.pt
Parameters: 121033
PyTorch input shape: (1, 1, 256, 256)
PyTorch input dtype: torch.float32
PyTorch output shape: (1, 1, 256, 256)
ONNX expected input shape: [1, 1, 256, 256]
ONNX input name: thermal_b10
ONNX FP32 output shape: (1, 1, 256, 256)

===== PYTORCH vs ONNX FP32 =====
Max absolute difference : 5.435943603515625e-05
Mean absolute difference: 1.5369514585472643e-06
FP32 numerical check: PASS
ONNX INT8 output shape: (1, 1, 256, 256)

===== ONNX FP32 vs INT8 =====
Max logit difference       : 4.541252136230469
Mean logit difference      : 0.6536070704460144
Max probability difference : 0.002071571536362171
Mean probability difference: 3.190856659784913e-06
INT8 numerical sanity check: PASS

===== CELL 19 RESULT =====
PyTorch vs FP32 ONNX : PASS
FP32 ONNX             : PASS
INT8 ONNX             : PASS
Model                 : BASE AFRICA
Parameters            : 121033
Input                 : [1, 1, 256

In [57]:
# ================================================================
# CELL 20 — HOST CPU INFERENCE BENCHMARK (EXPLICITLY NON-ONBOARD)
# ================================================================

import platform


def read_cpu_model():
    cpuinfo = Path('/proc/cpuinfo')
    if cpuinfo.exists():
        try:
            for line in cpuinfo.read_text(errors='ignore').splitlines():
                low = line.lower()
                if low.startswith('model name') or low.startswith('hardware') or low.startswith('model'):
                    return line.split(':', 1)[1].strip() if ':' in line else line.strip()
        except Exception:
            pass
    return platform.processor() or 'unknown'


def make_benchmark_session(model_path, threads=1):
    so = ort.SessionOptions()
    so.intra_op_num_threads = int(threads)
    so.inter_op_num_threads = 1
    so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    return ort.InferenceSession(
        str(model_path),
        sess_options=so,
        providers=['CPUExecutionProvider'],
    )


def benchmark_session(session, input_name, input_array, warmup=20, runs=100):
    for _ in range(warmup):
        session.run(None, {input_name: input_array})
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        session.run(None, {input_name: input_array})
        times.append((time.perf_counter() - t0) * 1000.0)
    return {
        'mean_ms': float(np.mean(times)),
        'p50_ms': float(np.percentile(times, 50)),
        'p95_ms': float(np.percentile(times, 95)),
        'p99_ms': float(np.percentile(times, 99)),
    }


bench_input = check_images[:1].numpy().astype(np.float32)
bench_threads = 1
cpu_arch = platform.machine()
cpu_model = read_cpu_model()
ort_version = ort.__version__
is_arm = cpu_arch.lower() in {'aarch64', 'arm64', 'armv7l'}
is_cortex_a72 = 'cortex-a72' in cpu_model.lower()
if is_cortex_a72:
    benchmark_scope = 'Cortex-A72 target hardware'
elif is_arm:
    benchmark_scope = 'ARM host (not confirmed Cortex-A72)'
else:
    benchmark_scope = 'non-ARM/Kaggle host CPU'

bench_sess_fp32 = make_benchmark_session(onnx_sim, threads=bench_threads)
bench_sess_int8 = make_benchmark_session(onnx_int8, threads=bench_threads)

fp32_bench = benchmark_session(
    bench_sess_fp32,
    bench_sess_fp32.get_inputs()[0].name,
    bench_input,
)
int8_bench = benchmark_session(
    bench_sess_int8,
    bench_sess_int8.get_inputs()[0].name,
    bench_input,
)

benchmark_df = pd.DataFrame([
    {
        'model': 'FP32 ONNX',
        'size_mb': size_mb(onnx_sim),
        **fp32_bench,
    },
    {
        'model': 'INT8 static ONNX',
        'size_mb': size_mb(onnx_int8),
        **int8_bench,
    },
])

benchmark_df['benchmark_scope'] = benchmark_scope
benchmark_df['cpu_arch'] = cpu_arch
benchmark_df['cpu_model'] = cpu_model
benchmark_df['onnxruntime_version'] = ort_version
benchmark_df['intra_op_threads'] = bench_threads
benchmark_df['inter_op_threads'] = 1

print('CPU architecture:', cpu_arch)
print('CPU model:', cpu_model)
print('Benchmark scope:', benchmark_scope)
print('ONNX Runtime:', ort_version)
print('Threads: intra_op=1, inter_op=1')

if not is_cortex_a72:
    print('⚠️ Do NOT cite these latency values as Cortex-A72 onboard latency.')
else:
    print('✅ This host identifies as Cortex-A72; this benchmark is suitable as an on-target CPU measurement.')

display(benchmark_df)
benchmark_df.to_csv(
    OUTPUT_DIR / 'onnx_cpu_benchmark_v6_1.csv',
    index=False,
)

CPU architecture: x86_64
CPU model: 85
Benchmark scope: non-ARM/Kaggle host CPU
ONNX Runtime: 1.30.0
Threads: intra_op=1, inter_op=1
⚠️ Do NOT cite these latency values as Cortex-A72 onboard latency.


,model,size_mb,mean_ms,p50_ms,p95_ms,p99_ms,benchmark_scope,cpu_arch,cpu_model,onnxruntime_version,intra_op_threads,inter_op_threads
0,FP32 ONNX,0.467730,24.661232,24.499869,26.219693,27.201850,non-ARM/Kaggle host CPU,x86_64,85,1.30.0,1,1
1,INT8 static ONNX,0.146699,36.293881,36.124180,39.767581,42.555919,non-ARM/Kaggle host CPU,x86_64,85,1.30.0,1,1


## 9. Compact alert prototype

This cell demonstrates the **data reduction interface** only: no full 256×256 mask is transmitted. A confirmed event is reduced to geolocation, bounding box, area, confidence, temperature and quality information plus a CRC.

This is a **prototype binary packet**, not the final CCSDS packet. It must later be replaced by the mission telemetry schema.

In [59]:
# ================================================================
# CELL 21 — COMPACT ALERT PACKET PROTOTYPE
# ================================================================

MAX_EVENTS = 6
MIN_COMPONENT_PIXELS = 2
MAX_PACKET_BYTES = 200


def event_geolocation(image_path, cx, cy):
    with rasterio.open(image_path) as src:
        x, y = rasterio.transform.xy(src.transform, cy, cx, offset='center')
        if src.crs is not None:
            lon, lat = rio_transform(
                src.crs,
                'EPSG:4326',
                [x],
                [y],
            )
            return float(lat[0]), float(lon[0])
    return float('nan'), float('nan')


def extract_events(probability, thermal_k, threshold, image_path):
    mask = probability >= threshold
    labels, n = ndimage.label(mask, structure=np.ones((3, 3), dtype=np.uint8))
    events = []
    for component_id in range(1, n + 1):
        ys, xs = np.where(labels == component_id)
        if len(xs) < MIN_COMPONENT_PIXELS:
            continue
        cx = int(round(xs.mean()))
        cy = int(round(ys.mean()))
        lat, lon = event_geolocation(image_path, cx, cy)
        vals_p = probability[labels == component_id]
        vals_t = thermal_k[labels == component_id]
        events.append({
            'lat_e7': int(round(lat * 1e7)) if np.isfinite(lat) else 0,
            'lon_e7': int(round(lon * 1e7)) if np.isfinite(lon) else 0,
            'x': int(xs.min()),
            'y': int(ys.min()),
            'w': int(xs.max() - xs.min() + 1),
            'h': int(ys.max() - ys.min() + 1),
            'area': int(len(xs)),
            'prob_u16': int(np.clip(round(float(vals_p.max()) * 65535), 0, 65535)),
            'temp_k_u16': int(np.clip(round(float(vals_t.mean())), 0, 65535)),
            'flags': 0,
        })
    events.sort(key=lambda e: e['prob_u16'], reverse=True)
    return events[:MAX_EVENTS]


def encode_alert(frame_id, timestamp_ms, events):
    # Header: magic, version, frame_id, timestamp, event_count, flags
    packet = bytearray(
        struct.pack('<BBIQBB', 0xF1, 1, int(frame_id), int(timestamp_ms), len(events), 0)
    )
    for e in events:
        packet.extend(
            struct.pack(
                '<iiHHHHHHHB',
                e['lat_e7'], e['lon_e7'],
                e['x'], e['y'], e['w'], e['h'],
                e['area'], e['prob_u16'],
                e['temp_k_u16'], e['flags'],
            )
        )
    crc = zlib.crc32(packet) & 0xFFFFFFFF
    packet.extend(struct.pack('<I', crc))
    return bytes(packet)

# Deterministic manual-test demonstration.
image_row = manual_test_df.iloc[0]

# Read directly from the Dataset.
# This avoids the multiprocessing DataLoader entirely.
image, _ = manual_test_ds[0]

# Dataset returns [1, H, W]; model expects [B, 1, H, W].
image = image.unsqueeze(0).to(device)

print("Demo image shape:", tuple(image.shape))

assert image.shape == (1, 1, 256, 256)

with torch.no_grad():
    probability = torch.sigmoid(final_model(image))[0, 0].cpu().numpy()

thermal_k = (
    image[0, 0].cpu().numpy()
    * (NORM_MAX_K - NORM_MIN_K)
    + NORM_MIN_K
)

events = extract_events(
    probability,
    thermal_k,
    MANUAL_FINAL_THRESHOLD,
    image_row.image_path,
)

# Mission rule: NO FIRE => NO ALERT PACKET.
packet = encode_alert(
    frame_id=1,
    timestamp_ms=int(time.time() * 1000),
    events=events,
) if events else b''

print('Events:', len(events))
print('Alert packet bytes:', len(packet))

if events:
    assert 0 < len(packet) <= MAX_PACKET_BYTES
else:
    assert len(packet) == 0

with open(OUTPUT_DIR / 'example_alert_packet.bin', 'wb') as f:
    f.write(packet)

print('Packet prototype rule verified: no fire -> 0 bytes; fire event(s) -> <=200 bytes.')

Demo image shape: (1, 1, 256, 256)
Events: 0
Alert packet bytes: 0
Packet prototype rule verified: no fire -> 0 bytes; fire event(s) -> <=200 bytes.


In [63]:
# ================================================================
# FINAL PAPER SUMMARY + MANUAL SAMPLE-SIZE DISCLOSURE
# ================================================================

# Derive these directly from the actual paired manual dataframe.
MATCHED_MANUAL_PATCHES = int(len(manual_df))
MATCHED_MANUAL_SCENES = int(manual_df["parent_scene"].nunique())

print("Matched manual patches:", MATCHED_MANUAL_PATCHES)
print("Matched manual scenes :", MATCHED_MANUAL_SCENES)

# ------------------------------------------------
# 1. Main model summary
# ------------------------------------------------
paper_summary = pd.DataFrame([
    {
        "model": "Tiny U-Net base, Africa held-out test",
        "parameters": int(sum(p.numel() for p in base_model.parameters())),
        "threshold": float(BASE_FINAL_THRESHOLD),
        "F1": float(base_test_metrics["f1"]),
        "IoU": float(base_test_metrics["iou"]),
        "Precision": float(base_test_metrics["precision"]),
        "Recall": float(base_test_metrics["recall"]),
        "FrameRecall": float(base_test_metrics["frame_recall"]),
        "FalseAlarmFrameRate": float(
            base_test_metrics["frame_false_alarm_rate"]
        ),
        "vs_baseline_F1": float(
            base_test_metrics["f1"] - BASELINE_F1
        ),
    },
    {
        "model": "Tiny U-Net manual fine-tuned, manual held-out test",
        "parameters": int(sum(p.numel() for p in final_model.parameters())),
        "threshold": float(MANUAL_FINAL_THRESHOLD),
        "F1": float(manual_test_metrics["f1"]),
        "IoU": float(manual_test_metrics["iou"]),
        "Precision": float(manual_test_metrics["precision"]),
        "Recall": float(manual_test_metrics["recall"]),
        "FrameRecall": float(manual_test_metrics["frame_recall"]),
        "FalseAlarmFrameRate": float(
            manual_test_metrics["frame_false_alarm_rate"]
        ),
        "vs_baseline_F1": float(
            manual_test_metrics["f1"] - BASELINE_F1
        ),
    },
])

display(paper_summary)

paper_summary.to_csv(
    OUTPUT_DIR / "FINAL_PAPER_TABLE_V6_1.csv",
    index=False,
)

# ------------------------------------------------
# 2. Manual sample-size disclosure
# ------------------------------------------------
# The split DataFrames are authoritative.
MATCHED_MANUAL_PATCHES = (
    len(manual_train_df)
    + len(manual_val_df)
    + len(manual_test_df)
)

MATCHED_MANUAL_SCENES = len(
    set(manual_train_scenes)
    | set(manual_val_scenes)
    | set(manual_test_scenes)
)

assert MATCHED_MANUAL_PATCHES == 100
assert MATCHED_MANUAL_SCENES == 10

manual_sample_summary = pd.DataFrame([{
    "matched_manual_patches": MATCHED_MANUAL_PATCHES,
    "matched_manual_scenes": MATCHED_MANUAL_SCENES,

    "manual_train_patches": len(manual_train_df),
    "manual_train_scenes": len(manual_train_scenes),

    "manual_val_patches": len(manual_val_df),
    "manual_val_scenes": len(manual_val_scenes),

    "manual_test_patches": len(manual_test_df),
    "manual_test_scenes": len(manual_test_scenes),

    "manual_sample_status": (
        "limited_small_scene_count"
        if MATCHED_MANUAL_SCENES < 6
        else "scene_count_reported"
    ),
}])

display(manual_sample_summary)

manual_sample_summary.to_csv(
    OUTPUT_DIR / "manual_sample_size_summary_v6_1.csv",
    index=False,
)

print("Matched manual patches:", MATCHED_MANUAL_PATCHES)
print("Matched manual scenes :", MATCHED_MANUAL_SCENES)

# ------------------------------------------------
# 3. Deployment manifest
# ------------------------------------------------
manifest = {
    "experiment": "Africa Tiny U-Net V6.1",

    "training_input": "Landsat-8 B10",
    "training_region": "Africa",
    "mask_definition": "algorithmic voting: >=2 methods agree",
    "scene_level_split": True,

    "baseline_f1": float(BASELINE_F1),

    "base_threshold": float(BASE_FINAL_THRESHOLD),
    "manual_threshold": float(MANUAL_FINAL_THRESHOLD),

    "parameters": int(
        sum(p.numel() for p in base_model.parameters())
    ),

    "normalization_min_k": float(NORM_MIN_K),
    "normalization_max_k": float(NORM_MAX_K),

    "manual_sample_size": (
        manual_sample_summary.iloc[0].to_dict()
    ),

    "artifacts": {
        "base_checkpoint": str(BEST_CHECKPOINT),
        "base_fp32_onnx": str(onnx_sim),
        "base_int8_onnx": str(onnx_int8),
    },

    "benchmark": benchmark_df.to_dict(
        orient="records"
    ),

    "alert_packet_prototype_max_bytes": int(
        MAX_PACKET_BYTES
    ),

    "pipeline_scope": {
        "gate_software_smoke_test": True,
        "flight_gate_1": True,
        "gate_2": True,
        "nuc_in_notebook_pipeline": False,
        "target_hardware_benchmark": bool(is_cortex_a72),
    },

    "note": (
        "Packet is a prototype binary interface, not final CCSDS telemetry. "
        "Gate smoke test is Landsat B10 software integration only. "
        "PyRawS/THRAWS raw-data validation is a separate cross-sensor experiment. "
        "Host CPU latency is not an onboard claim unless the benchmark host "
        "is confirmed Cortex-A72."
    ),
}

with open(
    OUTPUT_DIR / "deployment_manifest_v6_1.json",
    "w"
) as f:
    json.dump(manifest, f, indent=2)

print("Deployment manifest updated.")

print()
print("===== FINAL ARTIFACTS =====")
print("Output directory:", OUTPUT_DIR)
print("Base checkpoint:", BEST_CHECKPOINT)
print("Base FP32 ONNX:", onnx_sim)
print("Base INT8 ONNX:", onnx_int8)

print()
print("===== FINAL STATUS =====")
print("Main model: BASE AFRICA")
print("Parameters:", int(sum(
    p.numel() for p in base_model.parameters()
)))
print("Base threshold:", BASE_FINAL_THRESHOLD)
print("Africa test F1:", base_test_metrics["f1"])
print("Africa test IoU:", base_test_metrics["iou"])
print("Manual test F1:", manual_test_metrics["f1"])
print("Matched manual patches:", MATCHED_MANUAL_PATCHES)
print("Matched manual scenes:", MATCHED_MANUAL_SCENES)
print("Alert packet prototype: PASS")
print("Deployment manifest: WRITTEN")

Matched manual patches: 0
Matched manual scenes : 0


,model,parameters,threshold,F1,IoU,Precision,Recall,FrameRecall,FalseAlarmFrameRate,vs_baseline_F1
0,"Tiny U-Net base, Africa held-out test",121033,0.49,0.407636,0.255994,0.517540,0.336234,0.301064,0.117365,0.007636
1,"Tiny U-Net manual fine-tuned, manual held-out ...",121033,0.49,0.407638,0.255996,0.856909,0.267428,0.234043,0.000000,0.007638


,matched_manual_patches,matched_manual_scenes,manual_train_patches,manual_train_scenes,manual_val_patches,manual_val_scenes,manual_test_patches,manual_test_scenes,manual_sample_status
0,100,10,45,6,8,2,47,2,scene_count_reported


Matched manual patches: 100
Matched manual scenes : 10
Deployment manifest updated.

===== FINAL ARTIFACTS =====
Output directory: /kaggle/working/africa_tiny_unet_v6_1
Base checkpoint: /kaggle/working/africa_tiny_unet_v6_1/tiny_unet_b8_pw25_best.pt
Base FP32 ONNX: /kaggle/working/africa_tiny_unet_v6_1/tiny_unet_africa_base_fp32_simplified_v6_1.onnx
Base INT8 ONNX: /kaggle/working/africa_tiny_unet_v6_1/tiny_unet_africa_base_static_int8_v6_1.onnx

===== FINAL STATUS =====
Main model: BASE AFRICA
Parameters: 121033
Base threshold: 0.49
Africa test F1: 0.4076358015838806
Africa test IoU: 0.25599407597165563
Manual test F1: 0.4076383476227062
Matched manual patches: 100
Matched manual scenes: 10
Alert packet prototype: PASS
Deployment manifest: WRITTEN
